## Import

In [17]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, shape
from shapely import wkt
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import requests
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
import hdbscan
from sklearn.metrics import silhouette_score
import rasterio
from pyproj import Transformer
from shapely.geometry import box as shapely_box
from math import radians, cos, sin, asin, sqrt
from collections import Counter
import re
import shap
import optuna
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

DATA_ROOT = r"C:\Users\alexandre.batisse\.vscode\Projet\Use_Case_5_SCALIAN_x_COENERGY\data\Raw"


## Infos TABULA

In [18]:
def extract_tabula_data(filepath: str) -> dict:
    """Extrait automatiquement les consommations spécifiques du fichier TABULA."""
    df = pd.read_excel(filepath, header=0)
    
    tabula = {}
    renovation_states = ['not_renovated', 'partially_renovated', 'renovated']
    
    for block_idx, rs in enumerate(renovation_states):
        start = block_idx * 44
        end = start + 44
        
        for i in range(start, min(end, len(df))):
            row = df.iloc[i]
            bt_code = str(row.iloc[4]) if pd.notna(row.iloc[4]) else ''
            period = str(row.iloc[6]).strip() if pd.notna(row.iloc[6]) else ''
            val_15 = row.iloc[15]
            val_16 = row.iloc[16]
            
            if not bt_code or not period or pd.isna(val_15):
                continue
            
            # Exclure les sous-types (NBL, LightFrame)
            if 'NBL' in bt_code or '/' in bt_code:
                continue
            
            bt = bt_code.split('_')[0]
            heating = float(val_15)
            water = float(val_16) if pd.notna(val_16) else 0
            
            if bt not in tabula:
                tabula[bt] = {}
            if period not in tabula[bt]:
                tabula[bt][period] = {}
            tabula[bt][period][rs] = round(heating + water, 1)
    
    print(f"✓ TABULA extrait : {sum(len(p) for p in tabula.values())} combinaisons type×période")
    for bt in sorted(tabula.keys()):
        print(f"  {bt}: {len(tabula[bt])} périodes × {len(list(tabula[bt].values())[0])} états")
    
    return tabula

TABULA_SPECIFIC_HD = extract_tabula_data(r'C:\Users\alexandre.batisse\.vscode\Projet\Use_Case_5_SCALIAN_x_COENERGY\data\Raw\TABULA.xlsx')

✓ TABULA extrait : 36 combinaisons type×période
  EFH: 10 périodes × 3 états
  GMH: 5 périodes × 3 états
  HH: 2 périodes × 3 états
  MFH: 10 périodes × 3 états
  RH: 9 périodes × 3 états


## Geocodage

In [19]:
def geocode_address(address: str) -> dict:
    """
    Géocode une adresse via Nominatim (OpenStreetMap).
    Retourne lat, lon, adresse formatée, et les détails.
    """
    geolocator = Nominatim(user_agent="building_disaggregation_research")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    
    location = geocode(address, exactly_one=True, addressdetails=True)
    
    if location is None:
        print(f"❌ Adresse non trouvée : {address}")
        return None
    
    result = {
        'address_input': address,
        'address_found': location.address,
        'lat': location.latitude,
        'lon': location.longitude,
        'details': location.raw.get('address', {}),
    }
    
    print(f"✓ Géocodage réussi")
    print(f"  Adresse : {result['address_found']}")
    print(f"  Coordonnées : ({result['lat']:.6f}, {result['lon']:.6f})")
    
    return result

In [113]:
# ── Test avec une adresse ─────────────────────
# Remplace par une vraie adresse de ton dataset pour tester
# Baumberger Weg 150, Dusseldorf, Germany
#Husemannstraße 1, Gelsenkirchen, Germany
#Paul-Moor-Weg 5a, Bottrop, Germany
#Bestenweg 13, Gladbeck, Germany

TEST_ADDRESS = "Paul-Moor-Weg 5a, Bottrop, Germany"

geo_result = geocode_address(TEST_ADDRESS)
geo_result

✓ Géocodage réussi
  Adresse : 5a, Paul-Moor-Weg, Kirchhellen-Mitte, Kirchhellen, Bottrop, Nordrhein-Westfalen, 46244, Deutschland
  Coordonnées : (51.608990, 6.927741)


{'address_input': 'Paul-Moor-Weg 5a, Bottrop, Germany',
 'address_found': '5a, Paul-Moor-Weg, Kirchhellen-Mitte, Kirchhellen, Bottrop, Nordrhein-Westfalen, 46244, Deutschland',
 'lat': 51.6089903,
 'lon': 6.9277413,
 'details': {'house_number': '5a',
  'road': 'Paul-Moor-Weg',
  'suburb': 'Kirchhellen-Mitte',
  'town': 'Kirchhellen',
  'city': 'Bottrop',
  'state': 'Nordrhein-Westfalen',
  'ISO3166-2-lvl4': 'DE-NW',
  'postcode': '46244',
  'country': 'Deutschland',
  'country_code': 'de'}}

## Trouver dataset et City_config

In [81]:
# =============================================================================
# ANALYSE AUTOMATIQUE DU DATASET — Génère CITY_CONFIG et BUILDING_STATS
# =============================================================================

def analyze_dataset(df: pd.DataFrame, city_name: str = "Unknown") -> tuple[dict, dict]:
    """
    Analyse le dataset measured et génère automatiquement :
    - CITY_CONFIG : consommations spécifiques, heating distribution, ratios HS/FA
    - BUILDING_STATS : statistiques détaillées par building_type (seuils, distributions)
    
    Args:
        df: DataFrame du dataset measured (individuel)
        city_name: nom de la ville
    
    Returns:
        (city_config, building_stats)
    """
    
    print(f"{'=' * 60}")
    print(f"ANALYSE AUTOMATIQUE DU DATASET — {city_name}")
    print(f"{'=' * 60}")
    print(f"  {len(df)} bâtiments dans le dataset")
    
    # ── 1. Consommations spécifiques ───────────────────────────
    print("\n── Consommations spécifiques ──")
    df_a = df.copy()
    df_a['conso_specifique'] = df_a['initial_heat_demand'] / df_a['heated_space']
    df_a = df_a[(df_a['conso_specifique'] > 10) & (df_a['conso_specifique'] < 500)]
    
    conso_spec = {}
    for cyc in sorted(df_a['construction_year_class'].dropna().unique()):
        conso_spec[cyc] = {}
        for rs in ['not_renovated', 'partially_renovated', 'renovated']:
            subset = df_a[(df_a['construction_year_class'] == cyc) & (df_a['renovation_state'] == rs)]
            if len(subset) > 0:
                conso_spec[cyc][rs] = int(round(subset['conso_specifique'].median()))
            else:
                # Fallback : médiane globale de la classe
                fallback = df_a[df_a['construction_year_class'] == cyc]['conso_specifique'].median()
                conso_spec[cyc][rs] = int(round(fallback)) if pd.notna(fallback) else 100
        print(f"  {cyc}: NR={conso_spec[cyc]['not_renovated']}, PR={conso_spec[cyc]['partially_renovated']}, R={conso_spec[cyc]['renovated']}")
    
    # ── 2. Heating distribution par building_type ──────────────
    print("\n── Distribution heating system ──")
    heating_dist = {}
    if 'initial_heating_system' in df.columns:
        ct = pd.crosstab(df['building_type'], df['initial_heating_system'], normalize='index')
        for bt in ct.index:
            row = ct.loc[bt]
            top = row[row > 0.01].sort_values(ascending=False)
            heating_dist[bt] = {sys: int(round(pct * 100)) for sys, pct in top.head(6).items()}
            print(f"  {bt}: {heating_dist[bt]}")
    
    # ── 3. Ratio HS/FA par building_type ───────────────────────
    print("\n── Ratio heated_space / floor_area ──")
    df_a['ratio_hs_fa'] = df_a['heated_space'] / df_a['floor_area']
    hs_fa_ratio = {}
    for bt in df_a['building_type'].dropna().unique():
        subset = df_a[df_a['building_type'] == bt]
        ratio = subset['ratio_hs_fa'].median()
        hs_fa_ratio[bt] = round(ratio, 2) if pd.notna(ratio) else 1.0
        print(f"  {bt}: {hs_fa_ratio[bt]}")
    
    # ── 4. BUILDING_STATS — statistiques détaillées ────────────
    print("\n── Statistiques par building_type ──")
    building_stats = {}
    
    for bt in sorted(df['building_type'].dropna().unique()):
        subset = df[df['building_type'] == bt]
        stats = {
            'count': len(subset),
            'share_pct': round(len(subset) / len(df) * 100, 1),
        }
        
        # Stats pour chaque variable numérique clé
        for col in ['floor_area', 'heated_space', 'initial_heat_demand', 'construction_year']:
            if col in subset.columns:
                vals = subset[col].dropna()
                if len(vals) > 0:
                    stats[col] = {
                        'min': round(float(vals.min()), 1),
                        'p5': round(float(vals.quantile(0.05)), 1),
                        'p25': round(float(vals.quantile(0.25)), 1),
                        'median': round(float(vals.median()), 1),
                        'p75': round(float(vals.quantile(0.75)), 1),
                        'p95': round(float(vals.quantile(0.95)), 1),
                        'max': round(float(vals.max()), 1),
                    }
        
        # Stats pour construction_year_class (distribution)
        if 'construction_year_class' in subset.columns:
            cyc_dist = subset['construction_year_class'].value_counts(normalize=True)
            stats['construction_year_class_top3'] = {
                k: round(v * 100, 1) for k, v in cyc_dist.head(3).items()
            }
        
        # Stats pour renovation_state (distribution)
        if 'renovation_state' in subset.columns:
            rs_dist = subset['renovation_state'].value_counts(normalize=True)
            stats['renovation_state'] = {
                k: round(v * 100, 1) for k, v in rs_dist.items()
            }
        
        # Nombre d'appartements
        if 'number_of_apartments_min' in subset.columns:
            apt = subset['number_of_apartments_min'].dropna()
            if len(apt) > 0:
                stats['apartments'] = {
                    'median_min': int(apt.median()),
                    'median_max': int(subset['number_of_apartments_max'].dropna().median()) if 'number_of_apartments_max' in subset.columns else int(apt.median()),
                }
        # Ratio HS/FA comme proxy du nombre d'étages
        if 'heated_space' in subset.columns and 'floor_area' in subset.columns:
            ratio = subset['heated_space'] / subset['floor_area']
            ratio = ratio[(ratio > 0) & (ratio < 20)]
            if len(ratio) > 0:
                stats['ratio_hs_fa'] = {
                    'p25': round(float(ratio.quantile(0.25)), 2),
                    'median': round(float(ratio.median()), 2),
                    'p75': round(float(ratio.quantile(0.75)), 2),
                }
                
        building_stats[bt] = stats
        print(f"  {bt}: {stats['count']} bâtiments ({stats['share_pct']}%), "
              f"floor_area médian={stats.get('floor_area', {}).get('median', '?')}m²")
    
    # ── Construire le CITY_CONFIG ──────────────────────────────
    city_config = {
        "city_name": city_name,
        "conso_specifique": conso_spec,
        "heating_distribution": heating_dist,
        "hs_fa_ratio": hs_fa_ratio,
    }
    
    # Lister tous les systèmes de chauffage présents
    if 'initial_heating_system' in df.columns:
        all_systems = sorted(df['initial_heating_system'].dropna().unique().tolist())
        city_config['all_heating_systems'] = all_systems
        print(f"\n  Systèmes de chauffage : {all_systems}")
    
    print(f"\n{'=' * 60}")
    print(f"✓ Analyse terminée — {len(building_stats)} types de bâtiments analysés")
    
    return city_config, building_stats

In [82]:
# =============================================================================
# INITIALISATION AUTOMATIQUE PAR VILLE
# =============================================================================


def read_csv_auto(filepath: str) -> pd.DataFrame:
    """Lit un CSV en détectant automatiquement le séparateur."""
    # Lire les premières lignes pour détecter le séparateur
    with open(filepath, 'r', encoding='utf-8') as f:
        first_line = f.readline()
    
    # Compter les séparateurs potentiels
    sep_counts = {
        ',': first_line.count(','),
        ';': first_line.count(';'),
        '\t': first_line.count('\t'),
    }
    
    best_sep = max(sep_counts, key=sep_counts.get)
    print(f"  Séparateur détecté : '{repr(best_sep)}' ({sep_counts[best_sep]} occurrences)")
    
    df = pd.read_csv(filepath, sep=best_sep, encoding='utf-8')
    
    # Si une seule colonne, essayer avec un autre encodage
    if len(df.columns) <= 1:
        for enc in ['latin-1', 'iso-8859-1', 'cp1252']:
            try:
                df = pd.read_csv(filepath, sep=best_sep, encoding=enc)
                if len(df.columns) > 1:
                    print(f"  Encodage corrigé : {enc}")
                    break
            except:
                continue
    
    print(f"  ✓ {len(df)} lignes, {len(df.columns)} colonnes")
    return df

def initialize_city(city_name: str = None, data_folder: str = None, 
                    geo_details: dict = None) -> tuple[pd.DataFrame, pd.DataFrame, dict, dict, str]:
    """
    Initialise automatiquement la pipeline pour une ville donnée.
    
    Détecte la ville depuis le géocodage ou le nom fourni,
    charge les datasets, lance l'analyse, et retourne tout.
    
    Args:
        city_name: nom de la ville (optionnel si geo_details fourni)
        geo_details: dict retourné par geocode_address (pour auto-détection)
    
    Returns:
        (df_individual, df_blocks, city_config, building_stats, tif_folder)
    """
    # ── Auto-détection de la ville ─────────────────────────────
    if city_name is None and geo_details:
        city_name = geo_details.get('details', {}).get('city', 
                    geo_details.get('details', {}).get('town',
                    geo_details.get('details', {}).get('municipality', 'Unknown')))
    
    if city_name is None:
        raise ValueError("Impossible de déterminer la ville. Fournis city_name ou geo_details.")
    
    print(f"{'=' * 60}")
    print(f"INITIALISATION — {city_name}")
    print(f"{'=' * 60}")
    
    # ── Trouver le dossier de la ville ─────────────────────────
    if data_folder is None:
        data_folder = r"C:\Users\alexandre.batisse\.vscode\Projet\Use_Case_5_SCALIAN_x_COENERGY\data\Raw"
    
    city_folder = os.path.join(data_folder, city_name)
    
    if not os.path.isdir(city_folder):
        # Essayer sans accent / avec variantes
        for folder in os.listdir(data_folder):
            if city_name.lower() in folder.lower():
                city_folder = os.path.join(data_folder, folder)
                break
    
    if not os.path.isdir(city_folder):
        raise FileNotFoundError(f"Dossier non trouvé : {city_folder}")
    
    print(f"  Dossier : {city_folder}")
    
    # ── Trouver les fichiers CSV ───────────────────────────────
    csv_files = [f for f in os.listdir(city_folder) if f.endswith('.csv')]
    
    path_individual = None
    path_aggregated = None
    
    for f in csv_files:
        f_lower = f.lower()
        if 'aggregat' in f_lower or 'geo_area' in f_lower:
            path_aggregated = os.path.join(city_folder, f)
        elif 'building' in f_lower or 'twin' in f_lower:
            path_individual = os.path.join(city_folder, f)
        else:
            # Fallback : le plus gros fichier est probablement l'individuel
            if path_individual is None:
                path_individual = os.path.join(city_folder, f)
    
    print(f"  Dataset individuel : {os.path.basename(path_individual)}")
    print(f"  Dataset agrégé : {os.path.basename(path_aggregated) if path_aggregated else 'NON TROUVÉ'}")
    
    # ── Trouver le dossier TIF ─────────────────────────────────
    tif_folder = None
    for subfolder in ['height', 'tif', 'heights', 'TIF']:
        candidate = os.path.join(city_folder, subfolder)
        if os.path.isdir(candidate):
            tif_folder = candidate
            break
    
    if tif_folder:
        n_tif = len([f for f in os.listdir(tif_folder) if f.endswith('.tif')])
        print(f"  TIF : {tif_folder} ({n_tif} fichiers)")
    else:
        print(f"  ⚠️ Pas de dossier TIF trouvé")
    
    # ── Charger les datasets ───────────────────────────────────

    print(f"\n  Chargement des données...")
    df_individual = read_csv_auto(path_individual)
    print(f"  Dataset individuel : {len(df_individual)} bâtiments")
    
    df_blocks = None
    if path_aggregated:
        df_blocks = read_csv_auto(path_aggregated)
        print(f"  Dataset agrégé : {len(df_blocks)} blocs")
    
    # ── Nettoyer les données ───────────────────────────────────
    df_individual = df_individual.dropna(subset=['heated_space', 'initial_heat_demand'])

    df_individual = df_individual[(df_individual['heated_space'] > 0) & (df_individual['initial_heat_demand'] > 0)]
    print(f"  Après nettoyage : {len(df_individual)} bâtiments")
    
    # ── Analyser le dataset ────────────────────────────────────
    city_config, building_stats = analyze_dataset(df_individual, city_name)
    
    # ── Ajouter les coordonnées du centre-ville ────────────────
    CITY_CENTERS = {
        "Düsseldorf": (51.2277, 6.7735),
        "Gelsenkirchen": (51.5177, 7.0857),
    }
    center = CITY_CENTERS.get(city_name, None)
    if center:
        city_config["center_lat"] = center[0]
        city_config["center_lon"] = center[1]
    else:
        # Utiliser le centroïde des données comme approximation
        if 'geometry' in df_individual.columns:
            print(f"  ⚠️ Centre-ville non défini pour {city_name} — utilisation du centroïde des données")
        city_config["center_lat"] = 0
        city_config["center_lon"] = 0
    
    print(f"\n✓ Initialisation complète pour {city_name}")
    
    # Renommer la variable pour être générique
    CITY_CENTER = (city_config["center_lat"], city_config["center_lon"])

    return df_individual, df_blocks, city_config, building_stats, tif_folder


In [114]:
# ── Initialisation à partir du géocodage ───────────────────────

detected_city = geo_result['details'].get('city', 
                geo_result['details'].get('town',
                geo_result['details'].get('municipality', None)))

print(f"Ville détectée : {detected_city}")

df_individual, df_blocks, CITY_CONFIG, BUILDING_STATS, TIF_FOLDER = initialize_city(
    city_name=detected_city,
    data_folder=DATA_ROOT
)

# Mettre à jour les variables globales utilisées dans le profiling
CITY_CENTER = (CITY_CONFIG.get("center_lat", 0), CITY_CONFIG.get("center_lon", 0))
PATH_TIF = TIF_FOLDER

# Afficher ce qui a été créé
print(json.dumps(CITY_CONFIG, indent=2, ensure_ascii=False, default=str))
print("\n")

Ville détectée : Bottrop
INITIALISATION — Bottrop
  Dossier : C:\Users\alexandre.batisse\.vscode\Projet\Use_Case_5_SCALIAN_x_COENERGY\data\Raw\Bottrop
  Dataset individuel : Bottrop_buildings_enriched.csv
  Dataset agrégé : Bottrop_digital_twin_geo_area_aggregated.csv
  TIF : C:\Users\alexandre.batisse\.vscode\Projet\Use_Case_5_SCALIAN_x_COENERGY\data\Raw\Bottrop\height (102 fichiers)

  Chargement des données...
  Séparateur détecté : '','' (24 occurrences)
  ✓ 25076 lignes, 25 colonnes
  Dataset individuel : 25076 bâtiments
  Séparateur détecté : '';'' (48 occurrences)
  ✓ 1010 lignes, 49 colonnes
  Dataset agrégé : 1010 blocs
  Après nettoyage : 25076 bâtiments
ANALYSE AUTOMATIQUE DU DATASET — Bottrop
  25076 bâtiments dans le dataset

── Consommations spécifiques ──
  1860 - 1918: NR=130, PR=123, R=102
  1919 - 1948: NR=145, PR=116, R=97
  1949 - 1978: NR=142, PR=118, R=76
  1979 - 1986: NR=142, PR=113, R=85
  1987 - 1990: NR=138, PR=115, R=79
  1991 - 1995: NR=134, PR=107, R=97
  

## TIF 

In [31]:
def extract_height_from_tif(lat: float, lon: float, tif_folder: str) -> dict:
    """
    Extrait la hauteur du bâtiment à partir d'un dossier de tuiles TIF.
    Parcourt les tuiles pour trouver celle qui contient le point.
    
    Args:
        lat, lon: coordonnées GPS du bâtiment
        tif_folder: dossier contenant les fichiers .tif
    """
    
    if not os.path.isdir(tif_folder):
        print(f"⚠️ Dossier TIF non trouvé : {tif_folder}")
        return None
    
    tif_files = [f for f in os.listdir(tif_folder) if f.endswith('.tif')]
    if not tif_files:
        print(f"⚠️ Aucun fichier .tif dans {tif_folder}")
        return None
    
    print(f"  {len(tif_files)} tuiles TIF disponibles")
    
    # Transformer les coordonnées GPS → UTM (EPSG:25832 comme ton code existant)
    transformer_to_utm = Transformer.from_crs('EPSG:4326', 'EPSG:25832', always_xy=True)
    x_utm, y_utm = transformer_to_utm.transform(lon, lat)
    
    # Chercher la bonne tuile
    for tif_name in tif_files:
        tif_path = os.path.join(tif_folder, tif_name)
        
        try:
            with rasterio.open(tif_path) as src:
                bounds = src.bounds
                
                # Vérifier si le point est dans cette tuile
                if (bounds.left <= x_utm <= bounds.right and 
                    bounds.bottom <= y_utm <= bounds.top):
                    
                    print(f"  ✓ Tuile trouvée : {tif_name}")
                    
                    # Lire la hauteur au point (fenêtre 5x5 pour robustesse)
                    row, col = src.index(x_utm, y_utm)
                    
                    window_size = 5
                    half = window_size // 2
                    row_start = max(0, row - half)
                    col_start = max(0, col - half)
                    
                    window = rasterio.windows.Window(col_start, row_start, window_size, window_size)
                    data = src.read(1, window=window)
                    
                    # Filtrer nodata et valeurs invalides
                    nodata = src.nodata if src.nodata is not None else -9999
                    valid = data[(data != nodata) & (data > 0)]
                    
                    if len(valid) == 0:
                        print("  ⚠️ Pas de donnée de hauteur valide à cette position")
                        return {'height_m': None, 'estimated_floors': None, 'source': 'tif_no_data'}
                    
                    height_max = float(valid.max())
                    height_p90 = float(np.percentile(valid, 90))
                    
                    # Estimation étages (3.4m par étage, comme ton code)
                    FLOOR_HEIGHT_M = 3.4
                    estimated_floors = max(1, round(height_p90 / FLOOR_HEIGHT_M))
                    
                    result = {
                        'height_max_m': round(height_max, 2),
                        'height_p90_m': round(height_p90, 2),
                        'estimated_floors': estimated_floors,
                        'tif_file': tif_name,
                        'tif_crs': str(src.crs),
                        'tif_resolution_m': round(src.res[0], 2),
                        'source': 'tif'
                    }
                    
                    print(f"  Hauteur max : {result['height_max_m']:.1f} m")
                    print(f"  Hauteur p90 : {result['height_p90_m']:.1f} m")
                    print(f"  Étages estimés : {result['estimated_floors']}")
                    print(f"  Résolution : {result['tif_resolution_m']}m/pixel")
                    
                    return result
                    
        except Exception as e:
            continue
    
    print(f"  ⚠️ Aucune tuile ne couvre le point ({lat}, {lon})")
    return None


# ── Test ───────────────────────────────────────────────────────

if geo_result and os.path.isdir(TIF_FOLDER):
    height_data = extract_height_from_tif(geo_result['lat'], geo_result['lon'], TIF_FOLDER)
else:
    print("ℹ️ TIF non disponible — la hauteur sera estimée par OSM ou par le LLM")
    height_data = None  

  102 tuiles TIF disponibles
  ✓ Tuile trouvée : ndom50_32356_5719_1_nw_2022.tif
  Hauteur max : 10.1 m
  Hauteur p90 : 10.1 m
  Étages estimés : 3
  Résolution : 0.5m/pixel


## Identification bloc

In [11]:
def identify_block(lat: float, lon: float, df_blocks: pd.DataFrame) -> dict:
    """
    Identifie le bloc (floor) contenant le bâtiment.
    Utilise la colonne geometry du dataset agrégé.
    """
    point = Point(lon, lat)
    
    # Convertir les géométries texte en objets Shapely
    for idx, row in df_blocks.iterrows():
        geom_str = row.get('geometry', '')
        if not geom_str or pd.isna(geom_str):
            continue
        
        try:
            polygon = wkt.loads(geom_str)
            if polygon.contains(point):
                # Extraire les données du bloc
                block_data = row.to_dict()
                
                print(f"✓ Bloc identifié")
                print(f"  Floor ID : {row.get('floor_area_id', '?')}")
                print(f"  Nom : {row.get('floor_area_name', '?')}")
                print(f"  Nombre de bâtiments : {row.get('building_count', '?')}")
                
                # Afficher les proportions principales
                bt_cols = [c for c in df_blocks.columns if c.startswith('building_type__')]
                if bt_cols:
                    print(f"  Répartition building_type :")
                    for col in bt_cols:
                        val = row[col]
                        if val > 0.01:
                            type_name = col.replace('building_type__', '')
                            print(f"    {type_name}: {val*100:.1f}%")
                
                return block_data
        except Exception as e:
            continue
    
    print(f"❌ Aucun bloc trouvé pour ({lat}, {lon})")
    print(f"   → Le bâtiment est peut-être en dehors des limites des blocs")
    return None

## Contexte Spatial

In [ ]:


def haversine(lat1, lon1, lat2, lon2):
    """Distance en km entre deux points GPS."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * 6371 * asin(sqrt(a))


def compute_spatial_context(lat: float, lon: float, block_data: dict = None) -> dict:
    """
    Calcule les features de contexte spatial enrichies pour un bâtiment.
    
    Combine 3 sources :
    - Distance au centre-ville (géométrique)
    - Densité et types des bâtiments voisins (OSM)
    - Proportions du bloc agrégé (dataset CoEnergy)
    """
    result = {}
    
    # ── 1. Distance au centre-ville ────────────────────────────
    dist_center = haversine(lat, lon, CITY_CENTER[0], CITY_CENTER[1])
    result['distance_center_km'] = round(dist_center, 2)
    
    # ── 2. Densité et profil des bâtiments voisins (OSM) ──────
    print("  Récupération des bâtiments voisins (OSM)...")
    neighbors = get_nearby_buildings_profile(lat, lon, radius=200)
    result.update(neighbors)
    
    # ── 3. Proportions du bloc (si disponible) ─────────────────
    if block_data:
        result['block_building_count'] = block_data.get('building_count', None)

        # Répartition complète building_type
        bt_cols = {k.replace('building_type__', ''): round(v * 100, 1) 
                   for k, v in block_data.items() 
                   if k.startswith('building_type__') and isinstance(v, (int, float)) and v > 0.01}
        result['block_building_type_distribution'] = bt_cols
        
        # Répartition complète construction_year_class
        cyc_cols = {k.replace('construction_year_class__', ''): round(v * 100, 1)
                    for k, v in block_data.items() 
                    if k.startswith('construction_year_class__') and isinstance(v, (int, float)) and v > 0.01}
        result['block_construction_year_distribution'] = cyc_cols
        
        # Répartition complète heating_system
        hs_cols = {k.replace('initial_heating_system__', ''): round(v * 100, 1) 
                   for k, v in block_data.items() 
                   if k.startswith('initial_heating_system__') and isinstance(v, (int, float)) and v > 0.01}
        result['block_heating_distribution'] = hs_cols
        
        # Répartition complète renovation_state
        rs_cols = {k.replace('renovation_state__', ''): round(v * 100, 1) 
                   for k, v in block_data.items() 
                   if k.startswith('renovation_state__') and isinstance(v, (int, float)) and v > 0.01}
        result['block_renovation_distribution'] = rs_cols


    # ── Résumé ─────────────────────────────────────────────────
    print(f"✓ Contexte spatial calculé")
    print(f"  Distance centre-ville : {result['distance_center_km']:.2f} km")
    print(f"  Bâtiments dans 200m : {result.get('n_neighbors', '?')}")
    print(f"  Hauteur moy. voisins : {result.get('neighbor_avg_levels', '?')} niveaux")
    print(f"  Types voisins : {result.get('neighbor_building_types', '?')}")
    if block_data:
        print(f"  Répartition building_type dans le bloc : {result.get('block_building_type_distribution', '?')}")
        print(f"  Répartition construction_year_class dans le bloc : {result.get('block_construction_year_distribution', '?')}")
        print(f"  Répartition heating_system dans le bloc : {result.get('block_heating_distribution', '?')}")
        print(f"  Répartition renovation_state dans le bloc : {result.get('block_renovation_distribution', '?')}")
    
    return result


def get_nearby_buildings_profile(lat: float, lon: float, radius: int = 200) -> dict:
    """
    Récupère le profil des bâtiments voisins via OSM :
    - Nombre de bâtiments
    - Distribution des types (residential, commercial, industrial...)
    - Nombre moyen d'étages
    """
    overpass_url = "https://overpass-api.de/api/interpreter"
    query = f"""
    [out:json][timeout:15];
    way["building"](around:{radius},{lat},{lon});
    out tags;
    """

    max_retries = 5
    for attempt in range(max_retries):
        try:
            response = requests.post(
                overpass_url,
                data={'data': query},
                timeout=30,
                headers={'User-Agent': 'building_disaggregation_research/1.0'}
            )

            if response.status_code == 429 or response.status_code == 504:
                wait = 10 * (attempt + 1)
                print(f"  ⚠️ Overpass HTTP {response.status_code} — attente {wait}s (tentative {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue

            if response.status_code != 200:
                print(f"  ⚠️ Overpass HTTP {response.status_code} — tentative {attempt+1}/{max_retries}")
                time.sleep(5)
                continue

            if not response.text or len(response.text.strip()) == 0:
                print(f"  ⚠️ Réponse vide — tentative {attempt+1}/{max_retries}")
                time.sleep(5)
                continue

            data = response.json()
            elements = data.get('elements', [])

            if elements:
                break
            else:
                print(f"  ⚠️ Aucun élément retourné — tentative {attempt+1}/{max_retries}")
                time.sleep(5)
                continue

        except Exception as e:
            print(f"  ⚠️ Erreur : {e} — tentative {attempt+1}/{max_retries}")
            time.sleep(5)
            continue
    else:
        print(f"  ❌ Échec après {max_retries} tentatives")
        return {'n_neighbors': None, 'neighbor_building_types': {}}

    # Compter les types de bâtiments
    building_types = []
    levels_list = []

    for el in elements:
        tags = el.get('tags', {})
        bt = tags.get('building', 'yes')
        building_types.append(bt)

        levels = tags.get('building:levels')
        if levels:
            try:
                levels_list.append(float(levels))
            except ValueError:
                pass

    # Distribution des types
    type_counts = Counter(building_types)
    total = len(building_types)
    type_distribution = {k: round(v/total, 3) for k, v in type_counts.most_common(5)}

    result = {
        'n_neighbors': len(elements),
        'neighbor_building_types': type_distribution,
        'neighbor_dominant_type': type_counts.most_common(1)[0][0] if type_counts else 'unknown',
    }

    # Hauteur moyenne si disponible
    if levels_list:
        result['neighbor_avg_levels'] = round(np.mean(levels_list), 1)
        result['neighbor_max_levels'] = int(max(levels_list))
        result['neighbor_levels_available'] = len(levels_list)
    else:
        result['neighbor_avg_levels'] = None
        result['neighbor_max_levels'] = None
        result['neighbor_levels_available'] = 0

    return result

## Profiling

In [115]:
def collect_building_profile(address: str, df_blocks: pd.DataFrame, tif_path: str = None) -> dict:
    """
    Pipeline complète Phase 1 : adresse → profil complet du bâtiment.
    
    Retourne un dictionnaire avec toutes les données collectées,
    prêt à être passé au LLM pour prédiction.
    """
    print("=" * 60)
    print(f"PROFIL BÂTIMENT : {address}")
    print("=" * 60)
    
    profile = {'address': address}
    
    # 1. Géocodage
    print("\n── Étape 1 : Géocodage ──")
    geo = geocode_address(address)
    if geo is None:
        return None
    profile['geocoding'] = geo
    
    # 2. Empreinte bâtiment depuis le dataset individuel
    print("\n── Étape 2 : Empreinte bâtiment (dataset individuel) ──")
    footprint = None

    street_name = geo['details'].get('road') or geo['details'].get('street') or ''
    house_number = str(geo['details'].get('house_number', '')).strip()

    if 'street' in df_individual.columns:
        candidates = df_individual.copy()

        if street_name:
            street_lower = street_name.strip().lower()
            mask = pd.Series(False, index=candidates.index)
            if house_number:
                exact_address = f"{street_lower} {house_number}".strip()
                reverse_address = f"{house_number} {street_lower}".strip()
                mask |= candidates['street'].fillna('').str.lower().str.contains(exact_address, na=False)
                mask |= candidates['street'].fillna('').str.lower().str.contains(reverse_address, na=False)
            mask |= candidates['street'].fillna('').str.lower().str.contains(street_lower, na=False)
            candidates = candidates[mask]

        if len(candidates) == 0 and 'postal_code' in df_individual.columns and geo['details'].get('postcode'):
            candidates = df_individual[
                df_individual['postal_code'].astype(str).fillna('').str.contains(str(geo['details']['postcode']), na=False)
            ]

        if len(candidates) > 1 and 'geometry' in df_individual.columns:
            point = Point(geo['lon'], geo['lat'])
            candidates = candidates.copy()
            candidates['distance_to_point'] = candidates['geometry'].apply(
                lambda geom: Point(geo['lon'], geo['lat']).distance(wkt.loads(geom).centroid)
                if pd.notna(geom) else np.inf
            )
            candidates = candidates.sort_values('distance_to_point')

        if len(candidates) > 0:
            matched = candidates.iloc[0]
            footprint = {
                'source': 'dataset_match',
                'matched_street': matched.get('street'),
                'floor_area_m2': float(matched.get('floor_area')) if pd.notna(matched.get('floor_area')) else None,
                'geometry_wkt': matched.get('geometry'),
            }

    profile['footprint'] = footprint
    print(f"  Surface au sol : {footprint['floor_area_m2']:.1f} m²" if footprint and footprint.get('floor_area_m2') else "  Surface au sol : N/A")
    
# 3. Hauteur TIF (si disponible)
    print("\n── Étape 3 : Hauteur (TIF) ──")
    if tif_path and os.path.isdir(tif_path):
        height = extract_height_from_tif(geo['lat'], geo['lon'], tif_path)
        profile['height'] = height
    elif tif_path and os.path.isfile(tif_path):
        # Cas d'un fichier TIF unique
        height = extract_height_from_tif(geo['lat'], geo['lon'], os.path.dirname(tif_path))
        profile['height'] = height
    else:
        # Utiliser les niveaux OSM comme fallback
        osm_levels = footprint.get('osm_levels') if footprint else None
        if osm_levels:
            estimated_height = int(osm_levels) * 3.0
            profile['height'] = {
                'height_max_m': estimated_height,
                'estimated_floors': int(osm_levels),
                'source': 'osm_levels'
            }
            print(f"  ℹ️ Hauteur estimée via OSM levels : {estimated_height}m ({osm_levels} étages)")
        else:
            profile['height'] = None
            print(f"  ⚠️ Hauteur non disponible (tif_path={tif_path})")
    
    # 4. Identification du bloc
    print("\n── Étape 4 : Identification du bloc ──")
    block = identify_block(geo['lat'], geo['lon'], df_blocks)
    profile['block_data'] = block

    # 5. Contexte spatial
    print("\n── Étape 5 : Contexte spatial ──")
    spatial = compute_spatial_context(geo['lat'], geo['lon'], block_data=block) 
    profile['spatial_context'] = spatial
    
    # 6. Résumé
    print("\n" + "=" * 60)
    print("RÉSUMÉ DU PROFIL")
    print("=" * 60)
    print(f"  Adresse : {geo['address_found']}")
    print(f"  Coordonnées : ({geo['lat']:.6f}, {geo['lon']:.6f})")
    if footprint:
        print(f"  Surface au sol (OSM) : {footprint['floor_area_m2']:.1f} m²")
    if profile.get('height'):
        print(f"  Hauteur : {profile['height']['height_max_m']:.1f} m ({profile['height']['estimated_floors']} étages)")
    print(f"  Distance centre-ville : {spatial['distance_center_km']:.2f} km")
    if block:
        print(f"  Bloc : {block.get('floor_area_name', '?')} ({block.get('building_count', '?')} bâtiments)")
    
    return profile

# ── Test complet ───────────────────────────────────────────
profile = collect_building_profile(
    address=TEST_ADDRESS,
    df_blocks=df_blocks,
    tif_path=PATH_TIF if os.path.exists(PATH_TIF) else None
)

PROFIL BÂTIMENT : Paul-Moor-Weg 5a, Bottrop, Germany

── Étape 1 : Géocodage ──
✓ Géocodage réussi
  Adresse : 5a, Paul-Moor-Weg, Kirchhellen-Mitte, Kirchhellen, Bottrop, Nordrhein-Westfalen, 46244, Deutschland
  Coordonnées : (51.608990, 6.927741)

── Étape 2 : Empreinte bâtiment (dataset individuel) ──
  Surface au sol : 72.0 m²

── Étape 3 : Hauteur (TIF) ──
  102 tuiles TIF disponibles
  ✓ Tuile trouvée : ndom50_32356_5719_1_nw_2022.tif
  Hauteur max : 10.1 m
  Hauteur p90 : 10.1 m
  Étages estimés : 3
  Résolution : 0.5m/pixel

── Étape 4 : Identification du bloc ──
✓ Bloc identifié
  Floor ID : 9d9504b0-ed32-4049-9925-2ccae86d118d
  Nom : 71174.0
  Nombre de bâtiments : 51.0
  Répartition building_type :
    EFH: 47.1%
    GHD: 2.0%
    MFH: 27.5%
    RH: 13.7%
    Öffentlich: 9.8%

── Étape 5 : Contexte spatial ──
  Récupération des bâtiments voisins (OSM)...
✓ Contexte spatial calculé
  Distance centre-ville : 5775.43 km
  Bâtiments dans 200m : 538
  Hauteur moy. voisins : 1.3 

## Sauvegarde profil

In [116]:
# ── Sauvegarde du profil pour la Phase 2 ───────────────────
if profile:
    # Nettoyer pour JSON (retirer les objets non sérialisables)
    profile_clean = json.loads(json.dumps(profile, default=str))
    
    with open('building_profile_test.json', 'w') as f:
        json.dump(profile_clean, f, indent=2, ensure_ascii=False)

    print("✓ Profil sauvegardé dans building_profile_test.json")

✓ Profil sauvegardé dans building_profile_test.json


## Enrichissement du Dataset avec les hauteurs et les répartitions par bloc

In [117]:
def enrich_dataset_with_tif_fast(df: pd.DataFrame, tif_folder: str) -> pd.DataFrame:
    """Version optimisée — indexe les TIF d'abord puis extraction rapide."""
    
    df = df.copy()
    
    # 1. Indexer tous les TIF (bounds)
    print("Indexation des tuiles TIF...")
    tif_index = []
    tif_files = [f for f in os.listdir(tif_folder) if f.endswith('.tif')]
    
    for tif_name in tif_files:
        tif_path = os.path.join(tif_folder, tif_name)
        try:
            with rasterio.open(tif_path) as src:
                b = src.bounds
                tif_index.append({
                    'name': tif_name,
                    'path': tif_path,
                    'left': b.left, 'right': b.right,
                    'bottom': b.bottom, 'top': b.top,
                })
        except:
            continue
    
    print(f"  {len(tif_index)} tuiles indexées")
    
    # 2. Extraire les centroïdes et transformer en UTM
    print("Extraction des centroïdes...")
    transformer = Transformer.from_crs('EPSG:4326', 'EPSG:25832', always_xy=True)
    
    coords_utm = []
    for idx, row in df.iterrows():
        try:
            geom = wkt.loads(row['geometry'])
            c = geom.centroid
            x, y = transformer.transform(c.x, c.y)
            coords_utm.append((x, y))
        except:
            coords_utm.append((None, None))
    
    df['_x_utm'] = [c[0] for c in coords_utm]
    df['_y_utm'] = [c[1] for c in coords_utm]
    
    print(f"  {df['_x_utm'].notna().sum()} centroïdes calculés")
    
    # 3. Pour chaque tuile, extraire les hauteurs de tous les bâtiments dedans
    print("Extraction des hauteurs...")
    heights = pd.Series(index=df.index, dtype=float)
    floors = pd.Series(index=df.index, dtype=float)
    
    for i, tile in enumerate(tif_index):
        if i % 20 == 0:
            print(f"  Tuile {i}/{len(tif_index)}")
        
        # Filtrer les bâtiments dans cette tuile
        mask = (
            df['_x_utm'].notna() &
            (df['_x_utm'] >= tile['left']) & (df['_x_utm'] <= tile['right']) &
            (df['_y_utm'] >= tile['bottom']) & (df['_y_utm'] <= tile['top'])
        )
        
        buildings_in_tile = df[mask]
        if len(buildings_in_tile) == 0:
            continue
        
        try:
            with rasterio.open(tile['path']) as src:
                for idx, row in buildings_in_tile.iterrows():
                    try:
                        r, c = src.index(row['_x_utm'], row['_y_utm'])
                        window = rasterio.windows.Window(max(0,c-2), max(0,r-2), 5, 5)
                        data = src.read(1, window=window)
                        nodata = src.nodata if src.nodata is not None else -9999
                        valid = data[(data != nodata) & (data > 0)]
                        if len(valid) > 0:
                            h = float(np.percentile(valid, 90))
                            if h > 2.0:
                                heights[idx] = round(h, 2)
                                floors[idx] = max(1, round(h / 3.4))
                    except:
                        continue
        except:
            continue
    
    df['height_tif_m'] = heights
    df['floors_tif'] = floors
    
    # Nettoyage
    df = df.drop(columns=['_x_utm', '_y_utm'])
    
    valid = df['height_tif_m'].notna().sum()
    print(f"\n✓ Hauteurs extraites : {valid}/{len(df)} ({valid/len(df)*100:.0f}%)")
    
    return df

## FEATURE ENGINEERING AVEC TABULA ET DIGITAL TWIN

In [118]:
# =============================================================================
# FEATURE ENGINEERING — Préparation des données pour le ML
# =============================================================================

def get_tabula_specific_hd(building_type: str, construction_year_class: str, 
                            renovation_state: str, city_config: dict = None) -> float:
    PERIOD_MAP = {
        "1860 - 1918": "1860 ... 1918",
        "1919 - 1948": "1919 ... 1948",
        "1949 - 1978": "1969 ... 1978",
        "1979 - 1986": "1979 ... 1983",
        "1987 - 1990": "1984 ... 1994",
        "1991 - 1995": "1995 ... 2001",
        "1996 - 2000": "1995 ... 2001",
        "2001 - 2004": "2002 ... 2009",
        "2005 - 2008": "2002 ... 2009",
        "2009 - 2011": "2002 ... 2009",
        "2012 - 2023": "2002 ... 2009",
    }
    
    period = PERIOD_MAP.get(construction_year_class)
    
    # Essayer TABULA d'abord
    if period:
        bt_data = TABULA_SPECIFIC_HD.get(building_type, {})
        period_data = bt_data.get(period, {})
        value = period_data.get(renovation_state)
        if value is not None:
            return value
    
# Fallback 1 : mapping vers le type TABULA le plus proche
    TABULA_PROXY = {
        'HH': 'GMH',        # Tour → grand immeuble (plus proche en taille et usage)
        'GHD': 'MFH',       # Commerce → immeuble collectif (proxy par défaut)
        'Industrie': 'GMH',  # Industriel → grand bâtiment (grands volumes)
        'Öffentlich': 'MFH', # Public → immeuble collectif (taille moyenne)
    }
    
    if building_type not in TABULA_SPECIFIC_HD and period:
        proxy_type = TABULA_PROXY.get(building_type, 'MFH')
        proxy_data = TABULA_SPECIFIC_HD.get(proxy_type, {}).get(period, {})
        value = proxy_data.get(renovation_state)
        if value is not None:
            return value
            
    # Fallback 2 : médianes du dataset (CITY_CONFIG)
    if city_config:
        conso = city_config.get('conso_specifique', {}).get(construction_year_class, {})
        value = conso.get(renovation_state)
        if value is not None:
            return value
    
    # Fallback 3 : valeur par défaut
    return 100.0

def compute_block_tabula_proxy(block_data: dict, city_config: dict) -> float:
    """
    Calcule la consommation spécifique moyenne pondérée du bloc
    en croisant les distributions bt × cyc × rs avec TABULA.
    """
    bt_dist = {k.replace('building_type__', ''): v 
               for k, v in block_data.items() 
               if k.startswith('building_type__') and isinstance(v, (int, float)) and v > 0}
    
    cyc_dist = {k.replace('construction_year_class__', ''): v 
                for k, v in block_data.items() 
                if k.startswith('construction_year_class__') and isinstance(v, (int, float)) and v > 0}
    
    rs_dist = {k.replace('renovation_state__', ''): v 
               for k, v in block_data.items() 
               if k.startswith('renovation_state__') and isinstance(v, (int, float)) and v > 0}
    
    if not bt_dist or not cyc_dist or not rs_dist:
        return None
    
    weighted_sum = 0
    total_weight = 0
    
    for bt, bt_pct in bt_dist.items():
        for cyc, cyc_pct in cyc_dist.items():
            for rs, rs_pct in rs_dist.items():
                tabula_val = get_tabula_specific_hd(bt, cyc, rs, city_config)
                if tabula_val:
                    weight = bt_pct * cyc_pct * rs_pct
                    weighted_sum += tabula_val * weight
                    total_weight += weight
    
    return round(weighted_sum / total_weight, 1) if total_weight > 0 else None


# =============================================================================
# JOINTURE SPATIALE — Associer chaque bâtiment à son bloc
# =============================================================================

def assign_blocks_to_buildings(df_individual: pd.DataFrame, df_blocks: pd.DataFrame) -> pd.DataFrame:
    """
    Associe chaque bâtiment au bloc qui le contient via inclusion géométrique.
    Ajoute la colonne 'floor_area_id' au dataset individuel.
    """
    df = df_individual.copy()
    
    # Si floor_area_id existe déjà, pas besoin
    if 'floor_area_id' in df.columns:
        valid = df['floor_area_id'].notna().sum()
        print(f"✓ floor_area_id déjà présent : {valid}/{len(df)} bâtiments")
        return df
    
    print(f"Jointure spatiale : {len(df)} bâtiments × {len(df_blocks)} blocs...")
    
    # Préparer les polygones des blocs une seule fois
    block_polygons = []
    for idx, row in df_blocks.iterrows():
        geom_str = row.get('geometry', '')
        if not geom_str or pd.isna(geom_str):
            continue
        try:
            polygon = wkt.loads(geom_str)
            block_polygons.append({
                'floor_area_id': row['floor_area_id'],
                'polygon': polygon,
                'bounds': polygon.bounds,  # (minx, miny, maxx, maxy) pour filtrage rapide
            })
        except:
            continue
    
    print(f"  {len(block_polygons)} blocs avec géométrie valide")
    
    # Pour chaque bâtiment, trouver son bloc
    floor_area_ids = []
    
    for i, (idx, row) in enumerate(df.iterrows()):
        if i % 5000 == 0:
            print(f"  {i}/{len(df)} ({i/len(df)*100:.0f}%)")
        
        try:
            geom = wkt.loads(row['geometry'])
            centroid = geom.centroid
            point = Point(centroid.x, centroid.y)
            
            found = None
            for block in block_polygons:
                # Filtre rapide par bounding box
                bx_min, by_min, bx_max, by_max = block['bounds']
                if not (bx_min <= point.x <= bx_max and by_min <= point.y <= by_max):
                    continue
                # Test d'inclusion précis
                if block['polygon'].contains(point):
                    found = block['floor_area_id']
                    break
            
            floor_area_ids.append(found)
        except:
            floor_area_ids.append(None)
    
    df['floor_area_id'] = floor_area_ids
    
    matched = df['floor_area_id'].notna().sum()
    print(f"\n✓ Jointure terminée : {matched}/{len(df)} bâtiments associés ({matched/len(df)*100:.1f}%)")
    
    return df


def prepare_ml_features(df_individual: pd.DataFrame, df_blocks: pd.DataFrame, 
                        city_config: dict) -> pd.DataFrame:
    """
    Prépare le DataFrame avec toutes les features pour le ML.
    Deux sets de features :
    - full_features : toutes les données du dataset (cas idéal)
    - profile_features : seulement ce qui est disponible dans le profiling (comparable au LLM)
    """
    df_individual = assign_blocks_to_buildings(df_individual, df_blocks)

    df = df_individual.copy()
    
    # ── 1. Features dérivées ───────────────────────────────────
    # Ratio heated_space / floor_area (proxy étages)
    df['ratio_hs_fa'] = df['heated_space'] / df['floor_area']
    df['ratio_hs_fa'] = df['ratio_hs_fa'].clip(0.1, 20)  # Filtrer les aberrations

    df['floor_area_x_floors'] = df['floor_area'] * df['floors_tif'].fillna(1)
    df['building_volume'] = df['floor_area'] * df['height_tif_m'].fillna(3.4)

    # Distance au centre-ville (depuis la géométrie)
    center_lat = city_config.get('center_lat', 0)
    center_lon = city_config.get('center_lon', 0)
    
    if 'geometry' in df.columns:
        def extract_centroid(geom_str):
            try:
                geom = wkt.loads(geom_str)
                return geom.centroid.y, geom.centroid.x
            except:
                return None, None
        
        centroids = df['geometry'].apply(extract_centroid)
        df['building_lat'] = centroids.apply(lambda x: x[0] if x else None)
        df['building_lon'] = centroids.apply(lambda x: x[1] if x else None)
        
        df['distance_center_km'] = df.apply(
            lambda row: haversine(row['building_lat'], row['building_lon'], center_lat, center_lon)
            if pd.notna(row['building_lat']) else None, axis=1
        )
    
    # ── 2. Jointure avec les données du bloc ───────────────────
    if df_blocks is not None and 'floor_area_id' in df.columns:
        # Colonnes de proportions du bloc
        block_cols = [c for c in df_blocks.columns if '__' in c or c in ['building_count']]
        block_cols.append('floor_area_id')
        
        # Éviter les conflits de noms
        block_data = df_blocks[block_cols].copy()
        conflict_cols = [c for c in block_data.columns if c in df.columns and c != 'floor_area_id']
        block_data = block_data.rename(columns={c: f'block_{c}' for c in conflict_cols})
        
        df = df.merge(block_data, on='floor_area_id', how='left')
        print(f"  ✓ Jointure bloc : {len(block_cols)-1} colonnes ajoutées")
    
# Feature TABULA — consommation spécifique théorique
    df['tabula_specific_hd'] = df.apply(
        lambda row: get_tabula_specific_hd(
            row.get('building_type', ''),
            row.get('construction_year_class', ''),
            row.get('renovation_state', 'not_renovated')
        ), axis=1
    )
    print(f"  ✓ TABULA specific HD : {df['tabula_specific_hd'].notna().sum()}/{len(df)} valeurs")
    df['tabula_ihd_estimate'] = df['heated_space'] * df['tabula_specific_hd'].fillna(100)
    df['tabula_hd_not_renovated'] = df.apply(lambda row: get_tabula_specific_hd(row['building_type'], row['construction_year_class'], 'not_renovated', CITY_CONFIG) or 100, axis=1)
    df['tabula_hd_partially'] = df.apply(lambda row: get_tabula_specific_hd(row['building_type'], row['construction_year_class'], 'partially_renovated', CITY_CONFIG) or 80, axis=1)
    df['tabula_hd_renovated'] = df.apply(lambda row: get_tabula_specific_hd(row['building_type'], row['construction_year_class'], 'renovated', CITY_CONFIG) or 50, axis=1)


    # Calculer le proxy TABULA par bloc
    block_tabula = {}
    for floor_id in df_blocks['floor_area_id'].unique():
        block_row = df_blocks[df_blocks['floor_area_id'] == floor_id].iloc[0].to_dict()
        proxy = compute_block_tabula_proxy(block_row, city_config)
        block_tabula[floor_id] = proxy
    
    df['block_tabula_proxy'] = df['floor_area_id'].map(block_tabula)
    print(f"  ✓ Block TABULA proxy : {df['block_tabula_proxy'].notna().sum()}/{len(df)} valeurs")


    # ── 3. Nettoyage ───────────────────────────────────────────
    # Supprimer les lignes sans building_type (cible)
    df = df.dropna(subset=['building_type'])
    
    print(f"  ✓ Dataset préparé : {len(df)} bâtiments, {len(df.columns)} colonnes")
    print(f"  Distribution building_type :")
    for bt, count in df['building_type'].value_counts().items():
        print(f"    {bt}: {count} ({count/len(df)*100:.1f}%)")
    
    return df

## PIPELINE POUR ENRICHISSEMENT ET CREATION DU DATAFRAME ML

In [119]:
# =============================================================================
# PIPELINE COMPLÈTE POUR UN NOUVEAU DATASET
# =============================================================================

detected_city = geo_result['details'].get('city', 
                geo_result['details'].get('town',
                geo_result['details'].get('municipality', None)))

print(f"Ville détectée : {detected_city}")

city_name = detected_city

# 3. Jointure spatiale (ajoute floor_area_id si absent)
df_individual = assign_blocks_to_buildings(df_individual, df_blocks)

# 4. Enrichir avec les TIF (sur df_individual, PAS df_ml)
df_individual = enrich_dataset_with_tif_fast(df_individual, TIF_FOLDER)

# 5. Calculer les features dérivées qui dépendent du TIF
df_individual['floor_area_x_floors'] = df_individual['floor_area'] * df_individual['floors_tif'].fillna(1)
df_individual['building_volume'] = df_individual['floor_area'] * df_individual['height_tif_m'].fillna(3.4)

# 6. Sauvegarder le dataset enrichi
enriched_path = os.path.join(DATA_ROOT, city_name, f"{city_name}_buildings_enriched.csv")
df_individual.to_csv(enriched_path, index=False)
print(f"✓ Dataset enrichi sauvegardé : {enriched_path}")

# 7. Lancer prepare_ml_features
df_ml = prepare_ml_features(df_individual, df_blocks, CITY_CONFIG)

Ville détectée : Bottrop
✓ floor_area_id déjà présent : 25071/25076 bâtiments
Indexation des tuiles TIF...
  102 tuiles indexées
Extraction des centroïdes...
  25076 centroïdes calculés
Extraction des hauteurs...
  Tuile 0/102
  Tuile 20/102
  Tuile 40/102
  Tuile 60/102
  Tuile 80/102
  Tuile 100/102

✓ Hauteurs extraites : 24992/25076 (100%)
✓ Dataset enrichi sauvegardé : C:\Users\alexandre.batisse\.vscode\Projet\Use_Case_5_SCALIAN_x_COENERGY\data\Raw\Bottrop\Bottrop_buildings_enriched.csv
✓ floor_area_id déjà présent : 25071/25076 bâtiments
  ✓ Jointure bloc : 38 colonnes ajoutées
  ✓ TABULA specific HD : 25076/25076 valeurs
  ✓ Block TABULA proxy : 25071/25076 valeurs
  ✓ Dataset préparé : 25076 bâtiments, 73 colonnes
  Distribution building_type :
    EFH: 10285 (41.0%)
    MFH: 8827 (35.2%)
    RH: 4008 (16.0%)
    GHD: 917 (3.7%)
    Öffentlich: 383 (1.5%)
    GMH: 339 (1.4%)
    Industrie: 317 (1.3%)


## DÉFINITION DES FEATURES POUR MODELES INDIVIDUELS

In [120]:
# =============================================================================
# FEATURES PROPRES PAR VARIABLE — Sans data leakage
# =============================================================================

# Features de base communes (jamais une cible)
BASE_FEATURES = ['floor_area', 'solar_potential', 'building_count', 'ratio_hs_fa']

if 'distance_center_km' in df_ml.columns:
    BASE_FEATURES.append('distance_center_km')
if 'height_tif_m' in df_ml.columns:
    BASE_FEATURES.append('height_tif_m')
if 'floors_tif' in df_ml.columns:
    BASE_FEATURES.append('floors_tif')

# Colonnes du bloc par catégorie
block_bt_cols = [c for c in df_ml.columns if c.startswith('building_type__')]
block_cyc_cols = [c for c in df_ml.columns if c.startswith('construction_year_class__')]
block_hs_cols = [c for c in df_ml.columns if c.startswith('initial_heating_system__')]
block_rs_cols = [c for c in df_ml.columns if c.startswith('renovation_state__')]

# Features par variable cible
FEATURES_FOR = {
    'building_type': BASE_FEATURES + block_cyc_cols + block_hs_cols + block_rs_cols + block_bt_cols
                     + ['heated_space', 'construction_year', 'number_of_apartments_min', 'number_of_apartments_max'],
    # Exclure type_of_use (directement lié à building_type)
    
    'type_of_use': BASE_FEATURES + block_bt_cols + block_cyc_cols + block_hs_cols + block_rs_cols
                   + ['heated_space', 'construction_year', 'number_of_apartments_min', 'number_of_apartments_max'],
    # Exclure building_type
    
    'construction_year_class': BASE_FEATURES + block_bt_cols + block_hs_cols + block_rs_cols
                               + ['heated_space', 'number_of_apartments_min', 'number_of_apartments_max'],
    # Exclure construction_year ET toutes les colonnes construction_year_class__ du bloc
    
    'initial_heating_system': BASE_FEATURES + block_bt_cols + block_cyc_cols + block_rs_cols
                              + ['heated_space', 'construction_year', 'number_of_apartments_min', 'number_of_apartments_max'],
    # Exclure les colonnes heating_system__ du bloc (trop directement liées)
    
    'renovation_state': BASE_FEATURES + block_bt_cols + block_cyc_cols + block_hs_cols
                        + ['heated_space', 'construction_year', 'number_of_apartments_min', 'number_of_apartments_max'],
    # Exclure les colonnes renovation_state__ du bloc
    
    'heated_space': BASE_FEATURES + block_bt_cols + block_cyc_cols + block_hs_cols + block_rs_cols
                    + ['construction_year', 'number_of_apartments_min', 'number_of_apartments_max'],
    # Exclure initial_heat_demand (calculé depuis heated_space)
    
    'initial_heat_demand': BASE_FEATURES + block_bt_cols + block_cyc_cols + block_hs_cols + block_rs_cols
                           + ['heated_space', 'construction_year', 'number_of_apartments_min', 'number_of_apartments_max' , 'tabula_specific_hd'],
    # Inclure heated_space car en production on le connaîtra (ou on l'aura prédit avant)
    
    'number_of_apartments_min': BASE_FEATURES + block_bt_cols + block_cyc_cols + block_hs_cols + block_rs_cols
                                + ['heated_space', 'construction_year'],
    # Exclure number_of_apartments_max (directement lié)
    
    'number_of_apartments_max': BASE_FEATURES + block_bt_cols + block_cyc_cols + block_hs_cols + block_rs_cols
                                + ['heated_space', 'construction_year'],
    # Exclure number_of_apartments_min (directement lié)
    'construction_year': BASE_FEATURES + block_bt_cols + block_hs_cols + block_rs_cols
                         + ['heated_space', 'number_of_apartments_min', 'number_of_apartments_max'],
    
}

# Filtrer les colonnes qui existent réellement
for target, feats in FEATURES_FOR.items():
    FEATURES_FOR[target] = [f for f in feats if f in df_ml.columns]
    print(f"  {target:30s} : {len(FEATURES_FOR[target])} features")

  building_type                  : 44 features
  type_of_use                    : 44 features
  construction_year_class        : 32 features
  initial_heating_system         : 32 features
  renovation_state               : 41 features
  heated_space                   : 43 features
  initial_heat_demand            : 45 features
  number_of_apartments_min       : 42 features
  number_of_apartments_max       : 42 features
  construction_year              : 32 features


## ENTRAÎNEMENT XGBOOST CLASSIQUE 

In [121]:
# =============================================================================
# ENTRAÎNEMENT PROPRE — Un modèle par variable avec les bonnes features
# =============================================================================

models = {}
label_encoders = {}
results_summary = {}

# ── Variables catégorielles ────────────────────────────────────
categorical_targets = ['building_type', 'type_of_use', 'construction_year_class',
                       'initial_heating_system', 'renovation_state']

for target in categorical_targets:
    print(f"\n{'='*60}")
    print(f"TRAINING — {target}")
    print(f"{'='*60}")
    
    feats = FEATURES_FOR[target]
    
    le_var = LabelEncoder()
    y_var = le_var.fit_transform(df_ml[target].fillna('Unknown'))
    X = df_ml[feats].fillna(0)
    
    # Filtrer les classes avec trop peu de membres
    unique, counts = np.unique(y_var, return_counts=True)
    rare_classes = unique[counts < 2]
    if len(rare_classes) > 0:
        mask = ~np.isin(y_var, rare_classes)
        X = X[mask].reset_index(drop=True)
        y_filtered = df_ml[target].fillna('Unknown')[mask].reset_index(drop=True)
        # Re-encoder après filtrage
        le_var = LabelEncoder()
        y_var = le_var.fit_transform(y_filtered)
        print(f"  ⚠️ {len(rare_classes)} classes retirées (< 2 membres)")


    # for target in categorical_targets:
    #     weird = df_ml[target][df_ml[target].apply(lambda x: isinstance(x, float) or (isinstance(x, str) and '.' in x and len(x) > 10))]
    #     if len(weird) > 0:
    #         print(f"  ⚠️ {target} : {len(weird)} valeurs suspectes — {weird.unique()[:5]}")

    X_train, X_test, y_train_var, y_test_var = train_test_split(
        X, y_var, test_size=0.2, random_state=42, stratify=y_var
    )
    
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric='mlogloss', use_label_encoder=False,
    )
    model.fit(X_train, y_train_var, eval_set=[(X_test, y_test_var)], verbose=False)
    
    y_pred_var = model.predict(X_test)
    acc = accuracy_score(y_test_var, y_pred_var)
    
    print(f"  Accuracy : {acc*100:.1f}%")
    print(classification_report(y_test_var, y_pred_var, target_names=le_var.classes_))
    
    models[target] = model
    label_encoders[target] = le_var
    results_summary[target] = f"{acc*100:.1f}%"

# ── Variables numériques ───────────────────────────────────────
numerical_targets = ['heated_space', 'initial_heat_demand', 'construction_year',
                     'number_of_apartments_min', 'number_of_apartments_max']

for target in numerical_targets:
    print(f"\n{'='*60}")
    print(f"TRAINING — {target}")
    print(f"{'='*60}")
    
    feats = FEATURES_FOR[target]
    
    df_clean = df_ml.dropna(subset=[target])
    df_clean = df_clean[df_clean[target] > 0]
    # Filtrer les outliers extrêmes pour IHD
    if target == 'initial_heat_demand':
        q01 = df_clean[target].quantile(0.02)
        q99 = df_clean[target].quantile(0.98)
        before = len(df_clean)
        df_clean = df_clean[(df_clean[target] >= q01) & (df_clean[target] <= q99)]
        print(f"  Outliers filtrés : {before - len(df_clean)} ({q01:.0f} < IHD < {q99:.0f})")
        
        model = xgb.XGBClassifier(
            n_estimators=300, max_depth=8, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
            eval_metric='mlogloss', use_label_encoder=False,
            scale_pos_weight=None,  # Laisser XGBoost gérer
        )
        # Ajouter sample_weight pour surpondérer les classes rares
        class_counts = pd.Series(y_train_var).value_counts()
        weights = y_train_var.map(lambda x: len(y_train_var) / (len(class_counts) * class_counts[x]) if hasattr(y_train_var, 'map') else 1)
    
    X = df_clean[feats].fillna(0)
    y_var = df_clean[target]
    
    X_train, X_test, y_train_var, y_test_var = train_test_split(
        X, y_var, test_size=0.2, random_state=seed if 'seed' in dir() else 42
    )
    
    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
    )
    model.fit(X_train, y_train_var, eval_set=[(X_test, y_test_var)], verbose=False)
    
    y_pred_var = model.predict(X_test)
    mae = mean_absolute_error(y_test_var, y_pred_var)
    mape = np.mean(np.abs((y_test_var - y_pred_var) / y_test_var)) * 100
    r2 = r2_score(y_test_var, y_pred_var)
    
    print(f"  MAE  : {mae:.2f}")
    print(f"  MAPE : {mape:.1f}%")
    print(f"  R²   : {r2:.4f}")
    
    models[target] = model
    results_summary[target] = f"MAPE={mape:.1f}%, R²={r2:.4f}"

# ── Résumé ─────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RÉSUMÉ GLOBAL — Sans data leakage")
print(f"{'='*60}")

for target, result in results_summary.items():
    print(f"  {target:30s} : {result}")

print(df_ml['initial_heat_demand'].describe())
print(f"\nValeurs < 100 : {(df_ml['initial_heat_demand'] < 100).sum()}")
print(f"Valeurs > 1e7 : {(df_ml['initial_heat_demand'] > 1e7).sum()}")


TRAINING — building_type
  Accuracy : 89.0%
              precision    recall  f1-score   support

         EFH       0.86      0.90      0.88      2057
         GHD       0.90      0.86      0.88       183
         GMH       1.00      1.00      1.00        68
   Industrie       0.93      0.98      0.95        63
         MFH       1.00      1.00      1.00      1766
          RH       0.71      0.62      0.66       802
  Öffentlich       0.75      0.79      0.77        77

    accuracy                           0.89      5016
   macro avg       0.88      0.88      0.88      5016
weighted avg       0.89      0.89      0.89      5016


TRAINING — type_of_use
  Accuracy : 99.0%
                  precision    recall  f1-score   support

  Gewerbegebäude       0.85      0.90      0.88       194
Industriegebäude       0.94      0.91      0.92        65
     Wohngebäude       1.00      1.00      1.00      4693
Öffentliche Hand       0.73      0.62      0.67        64

        accuracy       

## CREATION DES FEATURES POUR LES MODELES ARCHITECTURE CASCADE

In [122]:
# =============================================================================
# CASCADE ML — Entraînement hiérarchique 
# =============================================================================

# ── Niveau 1 : variables de base ───────────────────────────────
LEVEL1_TARGETS = ['building_type', 'type_of_use', 'construction_year_class']

# Encodage des variables de Niveau 1
le_bt = LabelEncoder()
df_ml['building_type_encoded'] = le_bt.fit_transform(df_ml['building_type'].fillna('Unknown'))

le_tou = LabelEncoder()
df_ml['type_of_use_encoded'] = le_tou.fit_transform(df_ml['type_of_use'].fillna('Unknown'))

le_cyc = LabelEncoder()
df_ml['construction_year_class_encoded'] = le_cyc.fit_transform(df_ml['construction_year_class'].fillna('Unknown'))

# Features de prédiction de Niveau 1
LEVEL1_PRED_FEATURES = ['building_type_encoded', 'type_of_use_encoded', 'construction_year_class_encoded']

# ── Niveau 2 : variables dépendantes de Niveau 1 ─────────────
LEVEL2_TARGETS = ['initial_heating_system', 'renovation_state', 'number_of_apartments_min', 'number_of_apartments_max']

# Encodage des variables de Niveau 2
le_ihs = LabelEncoder()
df_ml['initial_heating_system_encoded'] = le_ihs.fit_transform(df_ml['initial_heating_system'].fillna('Unknown'))

le_rs = LabelEncoder()
df_ml['renovation_state_encoded'] = le_rs.fit_transform(df_ml['renovation_state'].fillna('Unknown'))

LEVEL2_PRED_FEATURES = ['initial_heating_system_encoded', 'renovation_state_encoded']

# ── NOUVEAU : Niveau 3 : heated_space SEUL (avec features de base + Niveau 1) ────
LEVEL3_TARGETS = ['heated_space']  # <-- MODIFICATION : SEULEMENT heated_space

# ── NOUVEAU : Niveau 4 : initial_heat_demand et construction_year (avec heated_space comme feature) ────
LEVEL4_TARGETS = ['initial_heat_demand', 'construction_year']  # <-- MODIFICATION : Séparé du Niveau 3

# =============================================================================
# DÉFINITION DES FEATURES PAR NIVEAU (CORRIGÉE)
# =============================================================================
FEATURES_CASCADE = {}

# Niveau 1 : inchangé
for target in LEVEL1_TARGETS:
    FEATURES_CASCADE[target] = list(dict.fromkeys(FEATURES_FOR[target]))

# Niveau 2 : features de base + prédictions Niveau 1
for target in LEVEL2_TARGETS:
    base = FEATURES_FOR[target]
    extra = [f for f in LEVEL1_PRED_FEATURES if f not in base]
    FEATURES_CASCADE[target] = list(dict.fromkeys(base + extra))


# Niveau 3 : features de base
for target in LEVEL3_TARGETS:
    if target == 'heated_space':
        FEATURES_CASCADE[target] = list(dict.fromkeys(FEATURES_FOR[target]))
    else:
        FEATURES_CASCADE[target] = list(dict.fromkeys(FEATURES_FOR[target])) + LEVEL1_PRED_FEATURES

# Niveau 4 : features de base + Niveau 1 + Niveau 2 + heated_space (NOUVEAU)
for target in LEVEL4_TARGETS:
    base = FEATURES_FOR[target]
    extra = LEVEL1_PRED_FEATURES + LEVEL2_PRED_FEATURES
    if target == 'initial_heat_demand':
        extra.append('heated_space')  # <-- MODIFICATION : Ajout de heated_space comme feature
    if target == 'construction_year':
        # Éviter la redondance avec construction_year_class
        extra = [f for f in extra if 'construction_year_class' not in f]
    FEATURES_CASCADE[target] = list(dict.fromkeys(base + extra))

# Afficher le résumé
print("Features par niveau :")
for target, feats in FEATURES_CASCADE.items():
    if target in LEVEL1_TARGETS:
        level = "L1"
    elif target in LEVEL2_TARGETS:
        level = "L2"
    elif target in LEVEL3_TARGETS:
        level = "L3"
    else:
        level = "L4"
    print(f"  [{level}] {target:30s} : {len(feats)} features")

Features par niveau :
  [L1] building_type                  : 44 features
  [L1] type_of_use                    : 44 features
  [L1] construction_year_class        : 32 features
  [L2] initial_heating_system         : 35 features
  [L2] renovation_state               : 44 features
  [L2] number_of_apartments_min       : 45 features
  [L2] number_of_apartments_max       : 45 features
  [L3] heated_space                   : 43 features
  [L4] initial_heat_demand            : 50 features
  [L4] construction_year              : 36 features


## ENTRAÎNEMENT XGBOOST EN CASCADE

In [123]:
# =============================================================================
# FONCTION POUR DÉTERMINER LES CONTRAINTES DE MONOTONICITÉ
# =============================================================================
def get_monotonic_constraints(target, features):
    constraints = [0] * len(features)
    if target == 'heated_space':
        if 'floor_area' in features:
            constraints[features.index('floor_area')] = 1
    elif target == 'initial_heat_demand':
        if 'heated_space' in features:
            constraints[features.index('heated_space')] = 1
        if 'construction_year' in features:
            constraints[features.index('construction_year')] = -1
        if 'tabula_specific_hd' in features:
            constraints[features.index('tabula_specific_hd')] = 1
    return tuple(constraints)

# =============================================================================
# ENTRAÎNEMENT CASCADE (VERSION CORRIGÉE)
# =============================================================================
models_cascade = {}
label_encoders_cascade = {}
results_cascade = {}

# ── Niveau 1 : Variables de base (inchangé) ─────────────────────
print("\n" + "=" * 60)
print("NIVEAU 1 — Variables de base")
print("=" * 60)

for target in LEVEL1_TARGETS:
    feats = FEATURES_CASCADE[target]
    le_var = LabelEncoder()
    y_var = le_var.fit_transform(df_ml[target].fillna('Unknown'))
    X = df_ml[feats].fillna(0)
    
    # Filtrer les classes avec trop peu de membres
    unique, counts = np.unique(y_var, return_counts=True)
    rare_classes = unique[counts < 2]
    if len(rare_classes) > 0:
        mask = ~np.isin(y_var, rare_classes)
        X = X[mask].reset_index(drop=True)
        y_filtered = df_ml[target].fillna('Unknown')[mask].reset_index(drop=True)
        # Re-encoder après filtrage
        le_var = LabelEncoder()
        y_var = le_var.fit_transform(y_filtered)
        print(f"  ⚠️ {len(rare_classes)} classes retirées (< 2 membres)")

    X_train, X_test, y_train_var, y_test_var = train_test_split(
        X, y_var, test_size=0.2, random_state=42, stratify=y_var
    )
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric='mlogloss', use_label_encoder=False,
    )
    model.fit(X_train, y_train_var, eval_set=[(X_test, y_test_var)], verbose=False)
    y_pred_var = model.predict(X_test)
    acc = accuracy_score(y_test_var, y_pred_var)
    print(f"  {target:30s} : {acc*100:.1f}%")
    models_cascade[target] = model
    label_encoders_cascade[target] = le_var
    results_cascade[target] = f"{acc*100:.1f}%"

# ── Niveau 2 : Variables dépendantes (+ prédictions Niveau 1) ────
print("\n" + "=" * 60)
print("NIVEAU 2 — Variables dépendantes")
print("=" * 60)

for target in LEVEL2_TARGETS:
    feats = FEATURES_CASCADE[target]
    if target in ['number_of_apartments_min', 'number_of_apartments_max']:
        # Régression
        df_clean = df_ml.dropna(subset=[target])
        df_clean = df_clean[df_clean[target] >= 0]
        X = df_clean[feats].fillna(0)
        y_var = df_clean[target]
        X_train, X_test, y_train_var, y_test_var = train_test_split(
            X, y_var, test_size=0.2, random_state=42
        )
        model = xgb.XGBRegressor(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
        )
    else:
        # Classification
        le_var = LabelEncoder()
        y_var = le_var.fit_transform(df_ml[target].fillna('Unknown'))
        X = df_ml[feats].fillna(0)
    
    # Filtrer les classes avec trop peu de membres
        unique, counts = np.unique(y_var, return_counts=True)
        rare_classes = unique[counts < 2]
        if len(rare_classes) > 0:
            mask = ~np.isin(y_var, rare_classes)
            X = X[mask].reset_index(drop=True)
            y_filtered = df_ml[target].fillna('Unknown')[mask].reset_index(drop=True)
            # Re-encoder après filtrage
            le_var = LabelEncoder()
            y_var = le_var.fit_transform(y_filtered)
            print(f"  ⚠️ {len(rare_classes)} classes retirées (< 2 membres)")


        X_train, X_test, y_train_var, y_test_var = train_test_split(
            X, y_var, test_size=0.2, random_state=42, stratify=y_var
        )
        model = xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
            eval_metric='mlogloss', use_label_encoder=False,
        )
    model.fit(X_train, y_train_var, eval_set=[(X_test, y_test_var)], verbose=False)
    y_pred_var = model.predict(X_test)

    if target in ['number_of_apartments_min', 'number_of_apartments_max']:
        mask_nonzero = y_test_var > 0
        if mask_nonzero.sum() > 0:
            mape = np.mean(np.abs((y_test_var[mask_nonzero] - y_pred_var[mask_nonzero]) / y_test_var[mask_nonzero])) * 100
        else:
            mape = 0
        r2 = r2_score(y_test_var, y_pred_var)
        print(f"  {target:30s} : MAPE={mape:.1f}%, R²={r2:.4f}")
        results_cascade[target] = f"MAPE={mape:.1f}%, R²={r2:.4f}"
    else:
        acc = accuracy_score(y_test_var, y_pred_var)
        print(f"  {target:30s} : {acc*100:.1f}%")
        label_encoders_cascade[target] = le_var
        results_cascade[target] = f"{acc*100:.1f}%"
    models_cascade[target] = model

# ── NOUVEAU : Niveau 3 : heated_space SEUL ───────────────────────
print("\n" + "=" * 60)
print("NIVEAU 3 — heated_space (features de base + Niveau 1)")
print("=" * 60)

for target in LEVEL3_TARGETS:
    feats = FEATURES_CASCADE[target]
    df_clean = df_ml.dropna(subset=[target])
    df_clean = df_clean[df_clean[target] > 0]
    X = df_clean[feats].fillna(0)
    y_var = df_clean[target]
    X_train, X_test, y_train_var, y_test_var = train_test_split(
        X, y_var, test_size=0.2, random_state=42
    )

    # CONTRAINTE : heated_space ↑ si floor_area ↑
    constraints = get_monotonic_constraints(target, feats)

    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        monotonic_constraints=constraints,  # <-- NOUVEAU
    )
    model.fit(X_train, y_train_var, eval_set=[(X_test, y_test_var)], verbose=False)
    y_pred_var = model.predict(X_test)
    mask_nonzero = y_test_var > 0
    if mask_nonzero.sum() > 0:
        mape = np.mean(np.abs((y_test_var[mask_nonzero] - y_pred_var[mask_nonzero]) / y_test_var[mask_nonzero])) * 100
    else:
        mape = 0
    r2 = r2_score(y_test_var, y_pred_var)
    print(f"  {target:30s} : MAPE={mape:.1f}%, R²={r2:.4f}")
    models_cascade[target] = model
    results_cascade[target] = f"MAPE={mape:.1f}%, R²={r2:.4f}"

# Prédire heated_space sur TOUT le dataset pour l'utiliser comme feature au Niveau 4
df_ml['heated_space_pred'] = models_cascade['heated_space'].predict(df_ml[FEATURES_CASCADE['heated_space']].fillna(0))

# ── NOUVEAU : Niveau 4 : initial_heat_demand et construction_year ────
print("\n" + "=" * 60)
print("NIVEAU 4 — Variables finales (+ heated_space)")
print("=" * 60)

for target in LEVEL4_TARGETS:
    feats = FEATURES_CASCADE[target]
    df_clean = df_ml.dropna(subset=[target])
    df_clean = df_clean[df_clean[target] > 0]

    if target == 'initial_heat_demand':
        q01 = df_clean[target].quantile(0.02)
        q99 = df_clean[target].quantile(0.98)
        before = len(df_clean)
        df_clean = df_clean[(df_clean[target] >= q01) & (df_clean[target] <= q99)]
        print(f"  Outliers filtrés : {before - len(df_clean)} ({q01:.0f} < IHD < {q99:.0f})")

    X = df_clean[feats].fillna(0)
    y_var = df_clean[target]
    X_train, X_test, y_train_var, y_test_var = train_test_split(
        X, y_var, test_size=0.2, random_state=42
    )

    # CONTRAINTES : IHD ↑ si heated_space ↑, IHD ↓ si construction_year ↑
    constraints = get_monotonic_constraints(target, feats)

    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        monotonic_constraints=constraints,  # <-- NOUVEAU
    )
    model.fit(
        X_train.values,
        y_train_var.values,
        eval_set=[(X_test.values, y_test_var.values)],
        verbose=False
    )
    y_pred_var = model.predict(X_test.values)
    mask_nonzero = y_test_var.values > 0
    if mask_nonzero.sum() > 0:
        mape = np.mean(np.abs((y_test_var.values[mask_nonzero] - y_pred_var[mask_nonzero]) / y_test_var.values[mask_nonzero])) * 100
    else:
        mape = 0
    r2 = r2_score(y_test_var, y_pred_var)
    print(f"  {target:30s} : MAPE={mape:.1f}%, R²={r2:.4f}")
    models_cascade[target] = model
    results_cascade[target] = f"MAPE={mape:.1f}%, R²={r2:.4f}"

# ── Résumé ─────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RÉSUMÉ CASCADE CORRIGÉE")
print(f"{'='*60}")
for target, result in results_cascade.items():
    if target in LEVEL1_TARGETS:
        level = "L1"
    elif target in LEVEL2_TARGETS:
        level = "L2"
    elif target in LEVEL3_TARGETS:
        level = "L3"
    else:
        level = "L4"
    print(f"  [{level}] {target:30s} : {result}")


NIVEAU 1 — Variables de base
  building_type                  : 89.0%
  type_of_use                    : 99.0%
  construction_year_class        : 53.0%

NIVEAU 2 — Variables dépendantes
  initial_heating_system         : 65.9%
  renovation_state               : 52.7%
  number_of_apartments_min       : MAPE=31.0%, R²=0.7605
  number_of_apartments_max       : MAPE=28.2%, R²=0.8368

NIVEAU 3 — heated_space (features de base + Niveau 1)
  heated_space                   : MAPE=3.1%, R²=0.7573

NIVEAU 4 — Variables finales (+ heated_space)
  Outliers filtrés : 1004 (5474 < IHD < 132614)
  initial_heat_demand            : MAPE=15.4%, R²=0.8447
  construction_year              : MAPE=1.3%, R²=0.2572

RÉSUMÉ CASCADE CORRIGÉE
  [L1] building_type                  : 89.0%
  [L1] type_of_use                    : 99.0%
  [L1] construction_year_class        : 53.0%
  [L2] initial_heating_system         : 65.9%
  [L2] renovation_state               : 52.7%
  [L2] number_of_apartments_min       : MAP

## CLUSTERING — Feature engineering par regroupement de bâtiments

In [124]:
# =============================================================================
# CLUSTERING — Feature engineering par regroupement de bâtiments
# =============================================================================

# Features à utiliser pour le clustering
# On utilise uniquement les features numériques disponibles en production
CLUSTERING_FEATURES = ['floor_area', 'distance_center_km', 'height_tif_m', 
                        'floors_tif', 'floor_area_x_floors', 'building_volume']

# Ajouter les proportions du bloc les plus importantes
CLUSTERING_FEATURES += [c for c in df_ml.columns if c.startswith('building_type__')]

# Filtrer les features qui existent
CLUSTERING_FEATURES = [f for f in CLUSTERING_FEATURES if f in df_ml.columns]

print(f"Features pour le clustering : {len(CLUSTERING_FEATURES)}")

# Préparer les données
X_cluster = df_ml[CLUSTERING_FEATURES].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f"Dataset : {X_scaled.shape[0]} bâtiments, {X_scaled.shape[1]} features")

Features pour le clustering : 13
Dataset : 25076 bâtiments, 13 features


## MÉTHODE 1 : K-Means

In [125]:
# =============================================================================
# MÉTHODE 1 : K-Means
# =============================================================================

# Trouver le nombre optimal de clusters
print("=" * 60)
print("K-Means — Recherche du nombre optimal de clusters")
print("=" * 60)

silhouette_scores = {}
inertias = {}

for k in [5, 8, 10, 15, 20, 25, 30]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels, sample_size=5000)
    silhouette_scores[k] = sil
    inertias[k] = kmeans.inertia_
    print(f"  k={k:3d} : silhouette={sil:.4f}, inertia={kmeans.inertia_:.0f}")

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\n  Meilleur k : {best_k} (silhouette={silhouette_scores[best_k]:.4f})")

# Appliquer avec le meilleur k
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_ml['cluster_kmeans'] = kmeans_final.fit_predict(X_scaled)

# Analyser les clusters
print(f"\nAnalyse des clusters K-Means (k={best_k}) :")
for cluster_id in sorted(df_ml['cluster_kmeans'].unique()):
    subset = df_ml[df_ml['cluster_kmeans'] == cluster_id]
    dominant_bt = subset['building_type'].mode().iloc[0] if len(subset) > 0 else '?'
    avg_fa = subset['floor_area'].mean()
    avg_ihd = subset['initial_heat_demand'].mean()
    print(f"  Cluster {cluster_id:2d} : {len(subset):5d} bâtiments | "
          f"dominant={dominant_bt:12s} | FA moy={avg_fa:.0f}m² | IHD moy={avg_ihd:.0f}")

K-Means — Recherche du nombre optimal de clusters
  k=  5 : silhouette=0.2138, inertia=201346
  k=  8 : silhouette=0.2076, inertia=152679
  k= 10 : silhouette=0.2372, inertia=128576
  k= 15 : silhouette=0.2149, inertia=99300
  k= 20 : silhouette=0.1926, inertia=84683
  k= 25 : silhouette=0.2054, inertia=74995
  k= 30 : silhouette=0.2175, inertia=67854

  Meilleur k : 10 (silhouette=0.2372)

Analyse des clusters K-Means (k=10) :
  Cluster  0 :  3821 bâtiments | dominant=EFH          | FA moy=133m² | IHD moy=22280
  Cluster  1 :  2771 bâtiments | dominant=MFH          | FA moy=204m² | IHD moy=29927
  Cluster  2 :  3920 bâtiments | dominant=RH           | FA moy=85m² | IHD moy=15190
  Cluster  3 :     3 bâtiments | dominant=Industrie    | FA moy=27599m² | IHD moy=1822528
  Cluster  4 :  5254 bâtiments | dominant=MFH          | FA moy=171m² | IHD moy=35947
  Cluster  5 :   509 bâtiments | dominant=Industrie    | FA moy=451m² | IHD moy=484374
  Cluster  6 :   801 bâtiments | dominant=MFH   

## MÉTHODE 2 : HDBSCAN (density-based, détecte les outliers)

In [126]:
# =============================================================================
# MÉTHODE 2 : HDBSCAN (density-based, détecte les outliers)
# =============================================================================

print("=" * 60)
print("HDBSCAN — Clustering par densité")
print("=" * 60)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom'
)
df_ml['cluster_hdbscan'] = clusterer.fit_predict(X_scaled)

n_clusters = df_ml['cluster_hdbscan'].nunique()
n_noise = (df_ml['cluster_hdbscan'] == -1).sum()
print(f"  Clusters trouvés : {n_clusters} (dont bruit : {n_noise} bâtiments)")

for cluster_id in sorted(df_ml['cluster_hdbscan'].unique()):
    if cluster_id == -1:
        continue
    subset = df_ml[df_ml['cluster_hdbscan'] == cluster_id]
    dominant_bt = subset['building_type'].mode().iloc[0]
    avg_ihd = subset['initial_heat_demand'].median()
    print(f"  Cluster {cluster_id:2d} : {len(subset):5d} bâtiments | "
          f"dominant={dominant_bt:12s} | IHD médian={avg_ihd:.0f}")

HDBSCAN — Clustering par densité
  Clusters trouvés : 97 (dont bruit : 8918 bâtiments)
  Cluster  0 :   467 bâtiments | dominant=Industrie    | IHD médian=34675
  Cluster  1 :   117 bâtiments | dominant=GHD          | IHD médian=22648
  Cluster  2 :   109 bâtiments | dominant=RH           | IHD médian=12575
  Cluster  3 :    92 bâtiments | dominant=EFH          | IHD médian=20443
  Cluster  4 :    69 bâtiments | dominant=EFH          | IHD médian=29088
  Cluster  5 :   112 bâtiments | dominant=MFH          | IHD médian=20565
  Cluster  6 :   154 bâtiments | dominant=MFH          | IHD médian=40940
  Cluster  7 :    66 bâtiments | dominant=MFH          | IHD médian=17774
  Cluster  8 :   626 bâtiments | dominant=MFH          | IHD médian=18173
  Cluster  9 :    82 bâtiments | dominant=EFH          | IHD médian=19552
  Cluster 10 :  3575 bâtiments | dominant=EFH          | IHD médian=15945
  Cluster 11 :    53 bâtiments | dominant=RH           | IHD médian=15526
  Cluster 12 :    54 bâti

## FEATURES DÉRIVÉES DU CLUSTERING

In [127]:
# =============================================================================
# FEATURES DÉRIVÉES DU CLUSTERING
# =============================================================================

def add_cluster_features(df: pd.DataFrame, cluster_col: str) -> pd.DataFrame:
    """
    Ajoute des features statistiques basées sur le cluster de chaque bâtiment.
    """
    df = df.copy()
    
    # Statistiques par cluster
    for var in ['initial_heat_demand', 'heated_space', 'floor_area']:
        if var in df.columns:
            cluster_stats = df.groupby(cluster_col)[var].agg(['mean', 'median', 'std'])
            cluster_stats.columns = [f'{cluster_col}_{var}_{stat}' for stat in ['mean', 'median', 'std']]
            df = df.merge(cluster_stats, left_on=cluster_col, right_index=True, how='left')
    
    # Taille du cluster
    cluster_size = df[cluster_col].value_counts()
    df[f'{cluster_col}_size'] = df[cluster_col].map(cluster_size)
    
    # Distance au centroïde du cluster (proxy de typicité)
    # Plus la distance est grande, plus le bâtiment est atypique
    
    return df

# Appliquer pour K-Means
df_ml = add_cluster_features(df_ml, 'cluster_kmeans')

# Compter les nouvelles features
new_cols = [c for c in df_ml.columns if 'cluster_kmeans' in c]
print(f"\nNouvelles features cluster : {len(new_cols)}")
for c in new_cols:
    print(f"  {c}")


Nouvelles features cluster : 11
  cluster_kmeans
  cluster_kmeans_initial_heat_demand_mean
  cluster_kmeans_initial_heat_demand_median
  cluster_kmeans_initial_heat_demand_std
  cluster_kmeans_heated_space_mean
  cluster_kmeans_heated_space_median
  cluster_kmeans_heated_space_std
  cluster_kmeans_floor_area_mean
  cluster_kmeans_floor_area_median
  cluster_kmeans_floor_area_std
  cluster_kmeans_size


## CREATION DES FEATURES PRODUCTION AVEC INFORMATIONS MANQUANTES

In [130]:
# =============================================================================
# FEATURES PRODUCTION — Uniquement ce qui est disponible sans le dataset measured
# =============================================================================

# ── Niveau 1 : variables de base ───────────────────────────────
LEVEL1_TARGETS = ['building_type', 'type_of_use', 'construction_year_class']

# Encodage des variables de Niveau 1
le_bt = LabelEncoder()
df_ml['building_type_encoded'] = le_bt.fit_transform(df_ml['building_type'].fillna('Unknown'))

le_tou = LabelEncoder()
df_ml['type_of_use_encoded'] = le_tou.fit_transform(df_ml['type_of_use'].fillna('Unknown'))

le_cyc = LabelEncoder()
df_ml['construction_year_class_encoded'] = le_cyc.fit_transform(df_ml['construction_year_class'].fillna('Unknown'))

# Features de prédiction de Niveau 1
LEVEL1_PRED_FEATURES = ['building_type_encoded', 'type_of_use_encoded', 'construction_year_class_encoded']

# ── Niveau 2 : variables dépendantes de Niveau 1 ─────────────
LEVEL2_TARGETS = ['initial_heating_system', 'number_of_apartments_min', 'number_of_apartments_max', 'renovation_state']

# Encodage des variables de Niveau 2
le_ihs = LabelEncoder()
df_ml['initial_heating_system_encoded'] = le_ihs.fit_transform(df_ml['initial_heating_system'].fillna('Unknown'))

le_rs = LabelEncoder()
df_ml['renovation_state_encoded'] = le_rs.fit_transform(df_ml['renovation_state'].fillna('Unknown'))

LEVEL2_PRED_FEATURES = ['initial_heating_system_encoded', 'renovation_state_encoded']

# ── NOUVEAU : Niveau 3 : heated_space SEUL (avec features de base) ────
LEVEL3_TARGETS = ['heated_space'] 

# Encodage variable de Niveau 3
le_hs = LabelEncoder()
df_ml['heated_space_encoded'] = le_hs.fit_transform(df_ml['heated_space'].fillna(-1).astype(str))

LEVEL3_PRED_FEATURES = ['heated_space_encoded'] 

# ── NOUVEAU : Niveau 4 : initial_heat_demand et construction_year (avec heated_space comme feature) ────
LEVEL4_TARGETS = ['initial_heat_demand', 'construction_year'] 

# Features publiques (disponibles pour n'importe quelle adresse)
PUBLIC_FEATURES = ['floor_area', 'building_count', 'distance_center_km',
                   'height_tif_m', 'floors_tif', 'floor_area_x_floors', 'building_volume', 'block_tabula_proxy']

# Ajouter solar_potential si disponible (donnée publique dans certains cas)
if 'solar_potential' in df_ml.columns:
    PUBLIC_FEATURES.append('solar_potential')

cluster_features = [c for c in df_ml.columns 
                    if c.startswith('cluster_kmeans_') and c != 'cluster_kmeans'
                    and 'heated_space_std' not in c and 'floor_area_median' not in c]  # Retirer les inutiles SHAP

TABULA_RANGE_FEATURES = ['tabula_hd_not_renovated', 'tabula_hd_partially', 'tabula_hd_renovated']

# Dans FEATURES_PRODUCTION pour IHD
# Retirer renovation_state_encoded, ajouter les 3 TABULA range
# Ajouter tabula_specific_hd
# if 'tabula_specific_hd' in df_ml.columns:
#     PUBLIC_FEATURES.append('tabula_specific_hd')

# Proportions des blocs venant du dataset aggregated "city"
block_bt_cols = [c for c in df_ml.columns if c.startswith('building_type__')]
block_cyc_cols = [c for c in df_ml.columns if c.startswith('construction_year_class__')]
block_hs_cols = [c for c in df_ml.columns if c.startswith('initial_heating_system__')]
block_rs_cols = [c for c in df_ml.columns if c.startswith('renovation_state__')]

ALL_BLOCK_COLS = block_bt_cols + block_cyc_cols + block_hs_cols + block_rs_cols

# =============================================================================
# FEATURES PRODUCTION PAR NIVEAU
# =============================================================================

FEATURES_PRODUCTION = {}

# Niveau 1 : uniquement features publiques + bloc
# Pas de données measured sauf floor_area
for target in LEVEL1_TARGETS:
    base = PUBLIC_FEATURES + ALL_BLOCK_COLS 
    # Retirer les colonnes du bloc liées à la cible
    # if target == 'building_type':
    #     base = [f for f in base if not f.startswith('building_type__')]
    # elif target == 'type_of_use':
    #     base = [f for f in base if not f.startswith('type_of_use__')]
    # elif target == 'construction_year_class':
    #     base = [f for f in base if not f.startswith('construction_year_class__')]
    FEATURES_PRODUCTION[target] = list(dict.fromkeys([f for f in base if f in df_ml.columns]))

            
# Niveau 2 : features publiques + bloc + prédictions Niveau 1
for target in LEVEL2_TARGETS:
    base = PUBLIC_FEATURES + ALL_BLOCK_COLS + LEVEL1_PRED_FEATURES + TABULA_RANGE_FEATURES
    # if target == 'initial_heating_system':
    #     base = [f for f in base if not f.startswith('initial_heating_system__')]
    # elif target == 'renovation_state':
    #     base = [f for f in base if not f.startswith('renovation_state__')]
    FEATURES_PRODUCTION[target] = list(dict.fromkeys([f for f in base if f in df_ml.columns]))

# Niveau 3 : features publiques + bloc + prédictions Niveau 1
for target in LEVEL3_TARGETS:
    base = PUBLIC_FEATURES + ALL_BLOCK_COLS + LEVEL1_PRED_FEATURES + LEVEL2_PRED_FEATURES  + TABULA_RANGE_FEATURES 
    FEATURES_PRODUCTION[target] = list(dict.fromkeys([f for f in base if f in df_ml.columns]))

# Niveau 4 : features publiques + bloc + prédictions Niveau 1 + Niveau 2 + heated_space prédit
for target in LEVEL4_TARGETS:
    base = PUBLIC_FEATURES + ALL_BLOCK_COLS + LEVEL1_PRED_FEATURES + LEVEL2_PRED_FEATURES 
    if target == 'initial_heat_demand':
        # base += cluster_features
        base.append('heated_space_encoded')
        # base = [f for f in base if not f.startswith('renovation_state_encoded')]
    if target == 'construction_year':
        base += ['construction_year_class_encoded'] + TABULA_RANGE_FEATURES
    FEATURES_PRODUCTION[target] = list(dict.fromkeys([f for f in base if f in df_ml.columns]))


# Résumé comparatif
print(f"{'='*70}")
print("COMPARAISON FEATURES : FULL vs PRODUCTION")
print(f"{'='*70}")
for target in list(FEATURES_CASCADE.keys()):
    n_full = len(FEATURES_CASCADE.get(target, []))
    n_prod = len(FEATURES_PRODUCTION.get(target, []))
    level = "L1" if target in LEVEL1_TARGETS else "L2" if target in LEVEL2_TARGETS else "L3" if target in LEVEL3_TARGETS else "L4" if target in LEVEL4_TARGETS else "?"
    print(f"  [{level}] {target:30s} : full={n_full} features | production={n_prod} features")

print("Features pour initial_heat_demand (production) :")
for f in FEATURES_PRODUCTION['initial_heat_demand']:
    print(f"  {f}")

COMPARAISON FEATURES : FULL vs PRODUCTION
  [L1] building_type                  : full=44 features | production=42 features
  [L1] type_of_use                    : full=44 features | production=42 features
  [L1] construction_year_class        : full=32 features | production=42 features
  [L2] initial_heating_system         : full=35 features | production=48 features
  [L2] renovation_state               : full=44 features | production=48 features
  [L2] number_of_apartments_min       : full=45 features | production=48 features
  [L2] number_of_apartments_max       : full=45 features | production=48 features
  [L3] heated_space                   : full=43 features | production=50 features
  [L4] initial_heat_demand            : full=50 features | production=48 features
  [L4] construction_year              : full=36 features | production=50 features
Features pour initial_heat_demand (production) :
  floor_area
  building_count
  distance_center_km
  height_tif_m
  floors_tif
  floor_ar

## ENTRAINEMENT AVEC FEATURES MANQUANTES + CASCADE 

In [131]:
# =============================================================================
# ENTRAÎNEMENT MODE PRODUCTION
# =============================================================================

# =============================================================================
# FONCTION POUR DÉTERMINER LES CONTRAINTES DE MONOTONICITÉ
# =============================================================================
def get_monotonic_constraints(target, features):
    constraints = [0] * len(features)
    if target == 'heated_space':
        if 'floor_area' in features:
            constraints[features.index('floor_area')] = 1
    elif target == 'initial_heat_demand':
        if 'heated_space' in features:
            constraints[features.index('heated_space')] = 1
        if 'construction_year' in features:
            constraints[features.index('construction_year')] = -1
        if 'tabula_specific_hd' in features:
            constraints[features.index('tabula_specific_hd')] = 1
    return tuple(constraints)

models_production = {}
label_encoders_production = {}
results_production = {}

# Niveau 1
print("\n" + "=" * 60)
print("PRODUCTION — NIVEAU 1")
print("=" * 60)

for target in LEVEL1_TARGETS:
    feats = FEATURES_PRODUCTION[target]
    le_var = LabelEncoder()
    y_var = le_var.fit_transform(df_ml[target].fillna('Unknown'))
    X = df_ml[feats].fillna(0)
    
    # Filtrer les classes avec trop peu de membres
    unique, counts = np.unique(y_var, return_counts=True)
    rare_classes = unique[counts < 2]
    if len(rare_classes) > 0:
        mask = ~np.isin(y_var, rare_classes)
        X = X[mask].reset_index(drop=True)
        y_filtered = df_ml[target].fillna('Unknown')[mask].reset_index(drop=True)
        # Re-encoder après filtrage
        le_var = LabelEncoder()
        y_var = le_var.fit_transform(y_filtered)
        print(f"  ⚠️ {len(rare_classes)} classes retirées (< 2 membres)")
    X_train, X_test, y_train_var, y_test_var = train_test_split(
        X, y_var, test_size=0.2, random_state=42, stratify=y_var
    )
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric='mlogloss', use_label_encoder=False,
    )
    model.fit(X_train, y_train_var, eval_set=[(X_test, y_test_var)], verbose=False)
    y_pred_var = model.predict(X_test)
    acc = accuracy_score(y_test_var, y_pred_var)
    print(f"  {target:30s} : {acc*100:.1f}%")
    models_production[target] = model
    label_encoders_production[target] = le_var
    results_production[target] = f"{acc*100:.1f}%"

# Niveau 2
print("\n" + "=" * 60)
print("PRODUCTION — NIVEAU 2")
print("=" * 60)

for target in LEVEL2_TARGETS:
    feats = FEATURES_PRODUCTION[target]
    if target in ['number_of_apartments_min', 'number_of_apartments_max']:
        df_clean = df_ml.dropna(subset=[target])
        df_clean = df_clean[df_clean[target] >= 0]
        X = df_clean[feats].fillna(0)
        y_var = df_clean[target]
        X_train, X_test, y_train_var, y_test_var = train_test_split(
            X, y_var, test_size=0.2, random_state=42
        )
        model = xgb.XGBRegressor(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
        )
        model.fit(X_train, y_train_var, eval_set=[(X_test, y_test_var)], verbose=False)
        y_pred_var = model.predict(X_test)
        mask_nz = y_test_var > 0
        mape = np.mean(np.abs((y_test_var[mask_nz] - y_pred_var[mask_nz]) / y_test_var[mask_nz])) * 100 if mask_nz.sum() > 0 else 0
        r2 = r2_score(y_test_var, y_pred_var)
        print(f"  {target:30s} : MAPE={mape:.1f}%, R²={r2:.4f}")
        results_production[target] = f"MAPE={mape:.1f}%, R²={r2:.4f}"
    else:
        le_var = LabelEncoder()
        y_var = le_var.fit_transform(df_ml[target].fillna('Unknown'))
        X = df_ml[feats].fillna(0)
        
        # Filtrer les classes avec trop peu de membres
        unique, counts = np.unique(y_var, return_counts=True)
        rare_classes = unique[counts < 2]
        if len(rare_classes) > 0:
            mask = ~np.isin(y_var, rare_classes)
            X = X[mask].reset_index(drop=True)
            y_filtered = df_ml[target].fillna('Unknown')[mask].reset_index(drop=True)
            # Re-encoder après filtrage
            le_var = LabelEncoder()
            y_var = le_var.fit_transform(y_filtered)
            print(f"  ⚠️ {len(rare_classes)} classes retirées (< 2 membres)")
        X_train, X_test, y_train_var, y_test_var = train_test_split(
            X, y_var, test_size=0.2, random_state=42, stratify=y_var
        )
        model = xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
            eval_metric='mlogloss', use_label_encoder=False,
        )
        model.fit(X_train, y_train_var, eval_set=[(X_test, y_test_var)], verbose=False)
        y_pred_var = model.predict(X_test)
        acc = accuracy_score(y_test_var, y_pred_var)
        print(f"  {target:30s} : {acc*100:.1f}%")
        label_encoders_production[target] = le_var
        results_production[target] = f"{acc*100:.1f}%"
    models_production[target] = model

# Niveau 3
print("\n" + "=" * 60)
print("PRODUCTION — NIVEAU 3")
print("=" * 60)

for target in LEVEL3_TARGETS:
    feats = FEATURES_PRODUCTION[target]
    df_clean = df_ml.dropna(subset=[target])
    df_clean = df_clean[df_clean[target] > 0]
    X = df_clean[feats].fillna(0)
    y_var = df_clean[target]
    X_train, X_test, y_train_var, y_test_var = train_test_split(
        X, y_var, test_size=0.2, random_state=42
    )
    constraints = get_monotonic_constraints(target, feats)
    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        monotone_constraints=constraints,
    )
    model.fit(X_train, y_train_var, eval_set=[(X_test, y_test_var)], verbose=False)
    y_pred_var = model.predict(X_test)
    mask_nz = y_test_var > 0
    mape = np.mean(np.abs((y_test_var[mask_nz] - y_pred_var[mask_nz]) / y_test_var[mask_nz])) * 100 if mask_nz.sum() > 0 else 0
    r2 = r2_score(y_test_var, y_pred_var)
    print(f"  {target:30s} : MAPE={mape:.1f}%, R²={r2:.4f}")
    models_production[target] = model
    results_production[target] = f"MAPE={mape:.1f}%, R²={r2:.4f}"

# Niveau 4
print("\n" + "=" * 60)
print("PRODUCTION — NIVEAU 4")
print("=" * 60)

for target in LEVEL4_TARGETS:
    feats = FEATURES_PRODUCTION[target]
    df_clean = df_ml.dropna(subset=[target])
    df_clean = df_clean[df_clean[target] > 0]
    if target == 'initial_heat_demand':
        q01 = df_clean[target].quantile(0.02)
        q99 = df_clean[target].quantile(0.98)
        before = len(df_clean)
        df_clean = df_clean[(df_clean[target] >= q01) & (df_clean[target] <= q99)]
        print(f"  Outliers filtrés : {before - len(df_clean)}")
    X = df_clean[feats].fillna(0)
    y_var = df_clean[target]
    X_train, X_test, y_train_var, y_test_var = train_test_split(
        X, y_var, test_size=0.2, random_state=42
    )
    constraints = get_monotonic_constraints(target, feats)
    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        monotone_constraints=constraints,
    )
    model.fit(X_train, y_train_var, eval_set=[(X_test, y_test_var)], verbose=False)
    y_pred_var = model.predict(X_test)
    mask_nz = y_test_var > 0
    mape = np.mean(np.abs((y_test_var[mask_nz] - y_pred_var[mask_nz]) / y_test_var[mask_nz])) * 100 if mask_nz.sum() > 0 else 0
    r2 = r2_score(y_test_var, y_pred_var)
    print(f"  {target:30s} : MAPE={mape:.1f}%, R²={r2:.4f}")
    models_production[target] = model
    results_production[target] = f"MAPE={mape:.1f}%, R²={r2:.4f}"
        

# Résumé comparatif
print(f"\n{'='*70}")
print("COMPARAISON : FULL FEATURES vs PRODUCTION")
print(f"{'='*70}")
print(f"  {'Variable':30s} | {'Full':30s} | {'Production':30s}")
print(f"  {'-'*30} | {'-'*30} | {'-'*30}")
for target in results_cascade:
    full = results_cascade.get(target, '?')
    prod = results_production.get(target, '?')
    print(f"  {target:30s} | {full:30s} | {prod:30s}")


PRODUCTION — NIVEAU 1
  building_type                  : 77.6%
  type_of_use                    : 96.4%
  construction_year_class        : 52.4%

PRODUCTION — NIVEAU 2
  initial_heating_system         : 66.2%
  number_of_apartments_min       : MAPE=31.6%, R²=0.7591
  number_of_apartments_max       : MAPE=29.1%, R²=0.8363
  renovation_state               : 50.9%

PRODUCTION — NIVEAU 3
  heated_space                   : MAPE=14.9%, R²=0.5009

PRODUCTION — NIVEAU 4
  Outliers filtrés : 1004
  initial_heat_demand            : MAPE=16.3%, R²=0.8346
  construction_year              : MAPE=0.4%, R²=0.9356

COMPARAISON : FULL FEATURES vs PRODUCTION
  Variable                       | Full                           | Production                    
  ------------------------------ | ------------------------------ | ------------------------------
  building_type                  | 89.0%                          | 77.6%                         
  type_of_use                    | 99.0%             

## ÉVALUATION : Impact du clustering sur les prédictions

In [132]:
# =============================================================================
# ÉVALUATION : Impact du clustering sur les prédictions
# =============================================================================

# Ajouter les features cluster aux features production pour IHD
FEATURES_PRODUCTION_CLUSTER = FEATURES_PRODUCTION.copy()

cluster_features = [c for c in df_ml.columns if c.startswith('cluster_kmeans_') and c != 'cluster_kmeans']
FEATURES_PRODUCTION_CLUSTER['initial_heat_demand'] = list(dict.fromkeys(
    FEATURES_PRODUCTION['initial_heat_demand'] + cluster_features + ['cluster_kmeans']
))

# Entraîner le modèle IHD avec les features cluster
feats = FEATURES_PRODUCTION_CLUSTER['initial_heat_demand']
df_clean = df_ml.dropna(subset=['initial_heat_demand'])
df_clean = df_clean[df_clean['initial_heat_demand'] > 0]

q01 = df_clean['initial_heat_demand'].quantile(0.02)
q99 = df_clean['initial_heat_demand'].quantile(0.98)
df_clean = df_clean[(df_clean['initial_heat_demand'] >= q01) & (df_clean['initial_heat_demand'] <= q99)]

X = df_clean[feats].fillna(0)
y = df_clean['initial_heat_demand']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_ihd_cluster = xgb.XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
)
model_ihd_cluster.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = model_ihd_cluster.predict(X_test)
mask_nz = y_test > 0
mape_cluster = np.mean(np.abs((y_test[mask_nz] - y_pred[mask_nz]) / y_test[mask_nz])) * 100
r2_cluster = r2_score(y_test, y_pred)


# Comparer avec le modèle sans cluster
print(f"\n{'='*60}")
print("COMPARAISON IHD : Sans cluster vs Avec cluster")
print(f"{'='*60}")

print(f"  Nombre Features IHD sans cluster : {len(FEATURES_PRODUCTION['initial_heat_demand'])}")
print(f"  Nombre Features IHD avec cluster : {len(FEATURES_PRODUCTION_CLUSTER['initial_heat_demand'])}")
print(f"  Features IHD sans cluster : {FEATURES_PRODUCTION['initial_heat_demand']}")
print(f"  Features IHD avec cluster : {FEATURES_PRODUCTION_CLUSTER['initial_heat_demand']}")

print("\n")

print(f"  IHD Sans cluster : MAPE={results_production.get('initial_heat_demand', '?')}")
print(f"  IHD Avec cluster : MAPE={mape_cluster:.1f}%, R²={r2_cluster:.4f}")

# Même test pour heated_space
FEATURES_PRODUCTION_CLUSTER['heated_space'] = list(dict.fromkeys(
    FEATURES_PRODUCTION['heated_space'] + cluster_features + ['cluster_kmeans']
))

feats_hs = FEATURES_PRODUCTION_CLUSTER['heated_space']
df_clean_hs = df_ml.dropna(subset=['heated_space'])
df_clean_hs = df_clean_hs[df_clean_hs['heated_space'] > 0]

X_hs = df_clean_hs[feats_hs].fillna(0)
y_hs = df_clean_hs['heated_space']

X_train_hs, X_test_hs, y_train_hs, y_test_hs = train_test_split(X_hs, y_hs, test_size=0.2, random_state=42)

model_hs_cluster = xgb.XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
)
model_hs_cluster.fit(X_train_hs, y_train_hs, eval_set=[(X_test_hs, y_test_hs)], verbose=False)

y_pred_hs = model_hs_cluster.predict(X_test_hs)
mask_nz_hs = y_test_hs > 0
mape_hs_cluster = np.mean(np.abs((y_test_hs[mask_nz_hs] - y_pred_hs[mask_nz_hs]) / y_test_hs[mask_nz_hs])) * 100
r2_hs_cluster = r2_score(y_test_hs, y_pred_hs)

print(f"\n{'='*60}")
print("COMPARAISON HS : Sans cluster vs Avec cluster")
print(f"{'='*60}")

print(f"  Nombre Features HS sans cluster : {len(FEATURES_PRODUCTION['heated_space'])}")
print(f"  Nombre Features HS avec cluster : {len(FEATURES_PRODUCTION_CLUSTER['heated_space'])}")
print(f"  Features HS sans cluster : {FEATURES_PRODUCTION['heated_space']}")
print(f"  Features HS avec cluster : {FEATURES_PRODUCTION_CLUSTER['heated_space']}")
print("\n")

print(f"  HS sans cluster : {results_production.get('heated_space', '?')}")
print(f"  HS avec cluster : MAPE={mape_hs_cluster:.1f}%, R²={r2_hs_cluster:.4f}")


COMPARAISON IHD : Sans cluster vs Avec cluster
  Nombre Features IHD sans cluster : 48
  Nombre Features IHD avec cluster : 59
  Features IHD sans cluster : ['floor_area', 'building_count', 'distance_center_km', 'height_tif_m', 'floors_tif', 'floor_area_x_floors', 'building_volume', 'block_tabula_proxy', 'solar_potential', 'building_type__EFH', 'building_type__GHD', 'building_type__GMH', 'building_type__Industrie', 'building_type__MFH', 'building_type__RH', 'building_type__Öffentlich', 'construction_year_class__1860 - 1918', 'construction_year_class__1919 - 1948', 'construction_year_class__1949 - 1978', 'construction_year_class__1979 - 1986', 'construction_year_class__1987 - 1990', 'construction_year_class__1991 - 1995', 'construction_year_class__1996 - 2000', 'construction_year_class__2001 - 2004', 'construction_year_class__2005 - 2008', 'construction_year_class__2009 - 2011', 'construction_year_class__2012 - 2023', 'initial_heating_system__Fernwärme (Bestand)', 'initial_heating_syst

## SÉLECTION D'ADRESSES DE TEST PAR BUILDING_TYPE

In [105]:
# =============================================================================
# SÉLECTION D'ADRESSES DE TEST PAR BUILDING_TYPE
# =============================================================================

def select_test_addresses_by_type(df: pd.DataFrame, n_per_type: int = 2, seed: int = 1200) -> pd.DataFrame:
    """
    Sélectionne des adresses de test variées pour chaque building_type.
    Pour chaque type, prend un petit et un grand bâtiment (par floor_area).
    """
    np.random.seed(seed)
    
    test_buildings = []
    
    for bt in sorted(df['building_type'].dropna().unique()):
        subset = df[df['building_type'] == bt].dropna(subset=['street', 'floor_area'])
        # Filtrer les adresses sans numéro (doit contenir au moins un chiffre)
        subset = subset[subset['street'].str.contains(r'\d', na=False)]
        
        if len(subset) < 2:
            test_buildings.append(subset)
            continue
        
        # Prendre un petit (p25) et un grand (p75) par floor_area
        small = subset[subset['floor_area'] <= subset['floor_area'].quantile(0.25)].sample(n=1, random_state=seed)
        large = subset[subset['floor_area'] >= subset['floor_area'].quantile(0.75)].sample(n=1, random_state=seed)
        
        test_buildings.append(pd.concat([small, large]))
    
    result = pd.concat(test_buildings)
    
    print(f"✓ {len(result)} bâtiments sélectionnés pour le test")
    print(f"{'='*80}")
    for bt in sorted(result['building_type'].unique()):
        subset = result[result['building_type'] == bt]
        for _, row in subset.iterrows():
            print(f"  {bt:12s} | {row['street']:40s} | floor_area: {row['floor_area']:.1f}m² | year: {int(row['construction_year'])}")
    
    return result


# Lancer la sélection
test_df = select_test_addresses_by_type(df_individual, n_per_type=2)

✓ 14 bâtiments sélectionnés pour le test
  EFH          | Tauschlagstraße 33                       | floor_area: 62.4m² | year: 1914
  EFH          | Allinghofstraße 75                       | floor_area: 113.4m² | year: 1950
  GHD          | Ellinghorster Straße 132                 | floor_area: 51.4m² | year: 1962
  GHD          | Roßheidestraße 6                         | floor_area: 1699.3m² | year: 1997
  GMH          | Uferstraße 22                            | floor_area: 292.4m² | year: 1891
  GMH          | Wilhelm-Olejnik-Straße 114               | floor_area: 595.2m² | year: 1981
  Industrie    | Stollenstraße 11                         | floor_area: 192.9m² | year: 1981
  Industrie    | Brüsseler Straße 65                      | floor_area: 1037.9m² | year: 2005
  MFH          | Helmutstraße 70a                         | floor_area: 88.2m² | year: 1872
  MFH          | Hövelweg 9                               | floor_area: 229.0m² | year: 1956
  RH           | Händelstraße 

In [106]:
# Vérifier que le matching fonctionne
for _, row in test_df.iterrows():
    street = row['street']
    match_by_index = df_ml[df_ml.index == row.name]
    match_by_street = df_ml[df_ml['street'] == street] if 'street' in df_ml.columns else pd.DataFrame()
    print(f"{street[:40]:40s} | by_index: {len(match_by_index)} | by_street: {len(match_by_street)}")

Tauschlagstraße 33                       | by_index: 1 | by_street: 1
Allinghofstraße 75                       | by_index: 1 | by_street: 1
Ellinghorster Straße 132                 | by_index: 1 | by_street: 1
Roßheidestraße 6                         | by_index: 1 | by_street: 1
Uferstraße 22                            | by_index: 1 | by_street: 1
Wilhelm-Olejnik-Straße 114               | by_index: 1 | by_street: 1
Stollenstraße 11                         | by_index: 1 | by_street: 1
Brüsseler Straße 65                      | by_index: 1 | by_street: 1
Helmutstraße 70a                         | by_index: 1 | by_street: 1
Hövelweg 9                               | by_index: 1 | by_street: 1
Händelstraße 44b                         | by_index: 1 | by_street: 1
Theodor-Heuss-Straße 123                 | by_index: 1 | by_street: 1
Frochtwinkel 11                          | by_index: 1 | by_street: 1
Wilhelmstraße 60                         | by_index: 1 | by_street: 1


## Trouver Ground Truth

In [76]:
def find_ground_truth(profile: dict, df_individual: pd.DataFrame) -> pd.Series:
    """
    Cherche le bâtiment correspondant dans le dataset individuel.
    
    Stratégie :
    1. Matcher par adresse (rue + numéro) — le plus fiable
    2. Affiner par floor_area si plusieurs résultats
    3. Fallback par floor_area_id + surface au sol
    """
    if profile is None:
        print("⚠️ Profil manquant")
        return None
    
    # ── Extraire rue et numéro depuis le géocodage ─────────────
    geo = profile['geocoding']
    details = geo.get('details', {})
    
    # Nominatim retourne road + house_number dans les details
    street_name = details.get('road', '')
    house_number = details.get('house_number', '')
    
    # Construire l'adresse telle qu'elle apparaît dans le dataset
    # Format dataset : "FriedrichstraÃŸe 61a"
    search_address = f"{street_name} {house_number}".strip()
    
    print(f"  Adresse recherchée : '{search_address}'")
    
    if 'street' not in df_individual.columns:
        print("⚠️ Colonne 'street' absente du dataset")
        return _fallback_by_area(profile, df_individual)
    
    # ── Normaliser pour gérer l'encodage ß / ÃŸ ───────────────
    def normalize_street(s):
        """Normalise une adresse pour la comparaison."""
        if pd.isna(s):
            return ''
        s = str(s).strip().lower()
        # Gérer les variantes d'encodage du ß
        s = s.replace('ß', 'ss')
        s = s.replace('ãŸ', 'ss')      # ÃŸ en minuscule
        s = s.replace('\u00c3\u009f', 'ss')  # bytes UTF-8 mal décodés
        # Gérer les autres caractères allemands courants
        s = s.replace('ä', 'ae').replace('ã¤', 'ae')
        s = s.replace('ö', 'oe').replace('ã¶', 'oe')
        s = s.replace('ü', 'ue').replace('ã¼', 'ue')
        # Supprimer les espaces multiples
        s = ' '.join(s.split())
        return s
    
    search_normalized = normalize_street(search_address)
    print(f"  Normalisée : '{search_normalized}'")
    
    # Normaliser la colonne street du dataset
    df_individual['_street_normalized'] = df_individual['street'].apply(normalize_street)
    
    # ── Match exact (rue + numéro) ─────────────────────────────
    exact_matches = df_individual[df_individual['_street_normalized'] == search_normalized]
    
    if len(exact_matches) == 1:
        match = exact_matches.iloc[0]
        print(f"\n✓ Match exact trouvé !")
        _print_match(match, profile)
        df_individual.drop(columns=['_street_normalized'], inplace=True)
        return match
    
    if len(exact_matches) > 1:
        print(f"  {len(exact_matches)} matchs exacts — affinage par floor_area")
        match = _refine_by_floor_area(exact_matches, profile)
        _print_match(match, profile)
        df_individual.drop(columns=['_street_normalized'], inplace=True)
        return match
    
    # ── Match partiel (même rue, numéro différent ou absent) ───
    street_only = normalize_street(street_name)
    number_only = house_number.strip().lower()
    
    print(f"  Pas de match exact — recherche partielle sur '{street_only}'")
    
    # Chercher tous les bâtiments de la même rue
    same_street = df_individual[
        df_individual['_street_normalized'].str.startswith(street_only)
    ]
    
    if len(same_street) > 0:
        print(f"  {len(same_street)} bâtiments trouvés dans la même rue")
        
        # Chercher le bon numéro dans la rue
        if number_only:
            with_number = same_street[
                same_street['_street_normalized'].str.contains(number_only, na=False)
            ]
            if len(with_number) > 0:
                match = _refine_by_floor_area(with_number, profile)
                print(f"\n✓ Match par rue + numéro partiel")
                _print_match(match, profile)
                df_individual.drop(columns=['_street_normalized'], inplace=True)
                return match
        
        # Fallback : même rue, plus proche par floor_area
        match = _refine_by_floor_area(same_street, profile)
        print(f"\n⚠️ Match approximatif (même rue, meilleur floor_area)")
        _print_match(match, profile)
        df_individual.drop(columns=['_street_normalized'], inplace=True)
        return match
    
    # ── Dernier fallback : bloc + floor_area ───────────────────
    print("  ⚠️ Rue non trouvée — fallback par bloc + floor_area")
    df_individual.drop(columns=['_street_normalized'], inplace=True)
    return _fallback_by_area(profile, df_individual)


def _refine_by_floor_area(candidates: pd.DataFrame, profile: dict) -> pd.Series:
    """Parmi plusieurs candidats, choisir le plus proche par floor_area."""
    osm_floor_area = profile.get('footprint', {}).get('floor_area_m2')
    
    if osm_floor_area and 'floor_area' in candidates.columns:
        candidates = candidates.copy()
        candidates['_fa_diff'] = abs(candidates['floor_area'] - osm_floor_area)
        best = candidates.nsmallest(1, '_fa_diff').iloc[0]
        return best
    
    return candidates.iloc[0]


def _fallback_by_area(profile: dict, df_individual: pd.DataFrame) -> pd.Series:
    """Fallback : chercher par floor_area_id + surface au sol."""
    block_id = profile.get('block_data', {}).get('floor_area_id')
    
    if block_id and 'floor_area_id' in df_individual.columns:
        candidates = df_individual[df_individual['floor_area_id'] == block_id]
        if len(candidates) > 0:
            return _refine_by_floor_area(candidates, profile)
    
    print("❌ Aucun match trouvé")
    return None


def _print_match(match: pd.Series, profile: dict):
    """Affiche les détails du match trouvé."""
    osm_floor_area = profile.get('footprint', {}).get('floor_area_m2')
    
    print(f"  Adresse dataset : {match.get('street', '?')}")
    if osm_floor_area and 'floor_area' in match.index:
        diff = abs(match['floor_area'] - osm_floor_area)
        print(f"  floor_area → OSM: {osm_floor_area:.1f} m² | Dataset: {match['floor_area']:.1f} m² | Écart: {diff:.1f} m²")
    
    for col in ['building_type', 'heated_space', 'initial_heat_demand', 
                 'construction_year', 'initial_heating_system', 'renovation_state']:
        if col in match.index:
            print(f"  {col} : {match[col]}")

## Comparer prédiciton et ground Truth

In [77]:
def compare_predictions(predictions: dict, ground_truth: pd.Series) -> pd.DataFrame:
    if ground_truth is None:
        print("⚠️ Pas de ground truth disponible")
        return None
    
    rows = []
    
    categorical_vars = ['building_type', 'type_of_use', 'construction_year_class',
                        'initial_heating_system', 'renovation_state']
    integer_vars = ['number_of_apartments_min', 'number_of_apartments_max']
    numeric_vars = ['heated_space', 'initial_heat_demand']
    
    all_vars = categorical_vars + integer_vars + numeric_vars
    
    for var in all_vars:
        pred = predictions.get(var, None)
        actual = ground_truth.get(var, None) if var in ground_truth.index else None
        
        # Vérifier que les deux valeurs existent
        if pred is None or actual is None or (isinstance(actual, float) and pd.isna(actual)):
            rows.append({'variable': var, 'prédit': str(pred), 'réel': str(actual), 
                        'match': '?', 'erreur_%': ''})
            continue
        
        if var in categorical_vars:
            pred_str = str(pred).strip()
            actual_str = str(actual).strip()
            match = '✓' if pred_str == actual_str else '✗'
            rows.append({'variable': var, 'prédit': pred_str, 'réel': actual_str, 
                        'match': match, 'erreur_%': ''})
        
        elif var in integer_vars:
            try:
                match = '✓' if int(float(pred)) == int(float(actual)) else '✗'
                rows.append({'variable': var, 'prédit': str(int(float(pred))), 
                            'réel': str(int(float(actual))), 'match': match, 'erreur_%': ''})
            except (ValueError, TypeError):
                rows.append({'variable': var, 'prédit': str(pred), 'réel': str(actual), 
                            'match': '?', 'erreur_%': ''})
        
        elif var in numeric_vars:
            try:
                pred_f = float(pred)
                actual_f = float(actual)
                if actual_f != 0:
                    error_pct = abs(pred_f - actual_f) / abs(actual_f) * 100
                else:
                    error_pct = 0
                match = '✓' if error_pct < 20 else '~' if error_pct < 50 else '✗'
                rows.append({'variable': var, 'prédit': f"{pred_f:.1f}", 
                            'réel': f"{actual_f:.1f}", 'match': match, 
                            'erreur_%': f"{error_pct:.1f}"})
            except (ValueError, TypeError):
                rows.append({'variable': var, 'prédit': str(pred), 'réel': str(actual), 
                            'match': '?', 'erreur_%': ''})
    
    df_comparison = pd.DataFrame(rows)
    
    # Résumé
    cat_results = df_comparison[df_comparison['variable'].isin(categorical_vars)]
    cat_correct = (cat_results['match'] == '✓').sum()
    cat_total = (cat_results['match'].isin(['✓', '✗'])).sum()
    
    int_results = df_comparison[df_comparison['variable'].isin(integer_vars)]
    int_correct = (int_results['match'] == '✓').sum()
    int_total = (int_results['match'].isin(['✓', '✗'])).sum()
    
    num_results = df_comparison[df_comparison['variable'].isin(numeric_vars)]
    num_errors = num_results['erreur_%'].replace('', np.nan).dropna().astype(float)
    
    print("=" * 70)
    print("COMPARAISON PRÉDICTIONS vs GROUND TRUTH")
    print("=" * 70)
    print(df_comparison.to_string(index=False))
    print(f"\nVariables catégorielles : {cat_correct}/{cat_total} correctes")
    print(f"Variables entières : {int_correct}/{int_total} correctes")
    if len(num_errors) > 0:
        print(f"Variables numériques : erreur moyenne = {num_errors.mean():.1f}%")
    
    return df_comparison

## BENCHMARK SUR LES 16 ADRESSES

In [107]:
# =============================================================================
# BENCHMARK 16 ADRESSES — MODE PRODUCTION (cascade réelle)
# =============================================================================

def benchmark_ml_16_production(test_df: pd.DataFrame, df_ml: pd.DataFrame,
                                models: dict, label_encoders: dict,
                                FEATURES: dict) -> pd.DataFrame:
    results = []

    def normalize(s):
        if pd.isna(s): return ''
        s = str(s).strip().lower()
        s = s.replace('ß', 'ss').replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue')
        s = s.replace('ãŸ', 'ss').replace('ã¤', 'ae').replace('ã¶', 'oe').replace('ã¼', 'ue')
        return s

    df_ml['_street_norm'] = df_ml['street'].apply(normalize) if 'street' in df_ml.columns else ''

    for i, (idx, row) in enumerate(test_df.iterrows()):
        street = row['street']
        street_norm = normalize(street)

        match = df_ml[df_ml['_street_norm'] == street_norm]
        if len(match) == 0:
            print(f"  ⚠️ [{i+1}] {street} — non trouvé")
            continue
        if len(match) > 1:
            match = match.copy()
            match['_fa_diff'] = abs(match['floor_area'] - row['floor_area'])
            match = match.nsmallest(1, '_fa_diff')

        # Copier la ligne pour injecter les prédictions
        building_row = match.iloc[0].copy()
        result = {'street': street}

        # ── NIVEAU 1 ──
        for target in ['building_type', 'type_of_use', 'construction_year_class']:
            if target in models and target in FEATURES:
                feats = FEATURES[target]
                X_input = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
                pred_encoded = models[target].predict(X_input)[0]
                pred = label_encoders[target].inverse_transform([pred_encoded])[0]
                real = str(row.get(target, ''))
                result[f'{target}_pred'] = pred
                result[f'{target}_real'] = real
                result[f'{target}_match'] = '✓' if str(pred) == real else '✗'
                # Injecter la prédiction pour la cascade
                building_row[f'{target}_encoded'] = pred_encoded

        # ── NIVEAU 2 ──
        for target in ['initial_heating_system', 'renovation_state']:
            if target in models and target in FEATURES:
                feats = FEATURES[target]
                X_input = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
                pred_encoded = models[target].predict(X_input)[0]
                pred = label_encoders[target].inverse_transform([pred_encoded])[0]
                real = str(row.get(target, ''))
                result[f'{target}_pred'] = pred
                result[f'{target}_real'] = real
                result[f'{target}_match'] = '✓' if str(pred) == real else '✗'
                building_row[f'{target}_encoded'] = pred_encoded

        for target in ['number_of_apartments_min', 'number_of_apartments_max']:
            if target in models and target in FEATURES:
                feats = FEATURES[target]
                X_input = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
                pred = round(float(models[target].predict(X_input)[0]))
                real = float(row.get(target, 0))
                # Post-processing
                bt = result.get('building_type_pred', '')
                if bt in ['GHD', 'Industrie', 'Öffentlich']:
                    pred = 0
                elif bt in ['EFH', 'RH']:
                    pred = max(1, min(2, pred))
                elif bt in ['MFH', 'GMH', 'HH']:
                    pred = max(3, pred)
                error = abs(pred - real) / real * 100 if real != 0 else (0 if pred == 0 else float('inf'))
                result[f'{target}_pred'] = pred
                result[f'{target}_real'] = round(real)
                result[f'{target}_error'] = round(error, 1)

        # ── NIVEAU 3 : heated_space ──
        if 'heated_space' in models and 'heated_space' in FEATURES:
            feats = FEATURES['heated_space']
            X_input = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
            pred = float(models['heated_space'].predict(X_input)[0])
            real = float(row.get('heated_space', 0))
            error = abs(pred - real) / real * 100 if real != 0 else 0
            result['heated_space_pred'] = round(pred, 1)
            result['heated_space_real'] = round(real, 1)
            result['heated_space_error'] = round(error, 1)
            # Injecter pour Niveau 4
            building_row['heated_space'] = pred
            building_row['heated_space_encoded'] = pred

        # ── NIVEAU 4 : IHD et construction_year ──
        for target in ['initial_heat_demand', 'construction_year']:
            if target in models and target in FEATURES:
                feats = FEATURES[target]
                X_input = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
                pred = float(models[target].predict(X_input)[0])
                real = float(row.get(target, 0))
                error = abs(pred - real) / real * 100 if real != 0 else 0
                result[f'{target}_pred'] = round(pred, 1)
                result[f'{target}_real'] = round(real, 1)
                result[f'{target}_error'] = round(error, 1)

        results.append(result)

    df_ml.drop(columns=['_street_norm'], inplace=True, errors='ignore')
    df_r = pd.DataFrame(results)

    # Affichage
    print("=" * 80)
    print("BENCHMARK ML PRODUCTION — 16 adresses (cascade réelle)")
    print("=" * 80)

    for var in ['building_type', 'type_of_use', 'construction_year_class',
               'initial_heating_system', 'renovation_state']:
        col = f'{var}_match'
        if col in df_r.columns:
            correct = (df_r[col] == '✓').sum()
            total = len(df_r)
            print(f"  {var:30s} : {correct}/{total} ({correct/total*100:.0f}%)")

    for var in ['heated_space', 'initial_heat_demand', 'number_of_apartments_min',
                'number_of_apartments_max', 'construction_year']:
        col = f'{var}_error'
        if col in df_r.columns:
            errors = df_r[col].dropna()
            errors = errors[errors != float('inf')]
            if len(errors) > 0:
                print(f"  {var:30s} : erreur moy={errors.mean():.1f}%, médiane={errors.median():.1f}%")

    print(f"\nDétail :")
    for _, r in df_r.iterrows():
        bt_m = r.get('building_type_match', '?')
        print(f"  {bt_m} {r['street'][:35]:35s} | "
              f"BT: {r.get('building_type_pred','?'):12s} vs {r.get('building_type_real','?'):12s} | "
              f"HS err: {r.get('heated_space_error','?')}% | "
              f"IHD err: {r.get('initial_heat_demand_error','?')}%")

    return df_r

benchmark_16 = benchmark_ml_16_production(test_df, df_ml, models_production, 
                                           label_encoders_production, FEATURES_PRODUCTION)

BENCHMARK ML PRODUCTION — 16 adresses (cascade réelle)
  building_type                  : 10/14 (71%)
  type_of_use                    : 13/14 (93%)
  construction_year_class        : 12/14 (86%)
  initial_heating_system         : 12/14 (86%)
  renovation_state               : 10/14 (71%)
  heated_space                   : erreur moy=32.1%, médiane=14.4%
  initial_heat_demand            : erreur moy=33.1%, médiane=19.1%
  number_of_apartments_min       : erreur moy=39.7%, médiane=0.0%
  number_of_apartments_max       : erreur moy=68.6%, médiane=0.0%
  construction_year              : erreur moy=0.8%, médiane=0.5%

Détail :
  ✗ Tauschlagstraße 33                  | BT: RH           vs EFH          | HS err: 20.9% | IHD err: 20.1%
  ✓ Allinghofstraße 75                  | BT: EFH          vs EFH          | HS err: 11.5% | IHD err: 2.9%
  ✓ Ellinghorster Straße 132            | BT: GHD          vs GHD          | HS err: 40.5% | IHD err: 36.7%
  ✓ Roßheidestraße 6                    | BT: 

## ANALYSE IHD EN FONCTION BUILDING TYPE / IHS

In [ ]:
# =============================================================================
# ANALYSE IHD PAR BUILDING_TYPE ET HEATING_SYSTEM
# =============================================================================

def analyze_ihd_by_category(df_ml: pd.DataFrame, models: dict, label_encoders: dict,
                             FEATURES: dict, sample_per_category: int = 500):
    """
    Analyse la précision de IHD en fonction du building_type et du heating_system.
    Prédit IHD en mode cascade sur un échantillon de chaque catégorie.
    """
    
    def normalize(s):
        if pd.isna(s): return ''
        return str(s).strip().lower().replace('ß', 'ss').replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue')
    
    def predict_ihd_cascade(building_row_orig):
        """Prédit IHD en cascade pour une ligne du dataset."""
        building_row = building_row_orig.copy()
        
        # Niveau 1
        for target in ['building_type', 'type_of_use', 'construction_year_class']:
            if target in models and target in FEATURES:
                feats = FEATURES[target]
                X = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
                pred_enc = models[target].predict(X)[0]
                building_row[f'{target}_encoded'] = pred_enc
        
        # Niveau 2
        for target in ['initial_heating_system', 'renovation_state']:
            if target in models and target in FEATURES:
                feats = FEATURES[target]
                X = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
                pred_enc = models[target].predict(X)[0]
                building_row[f'{target}_encoded'] = pred_enc
        
        # Niveau 3 : heated_space
        if 'heated_space' in models and 'heated_space' in FEATURES:
            feats = FEATURES['heated_space']
            X = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
            hs_pred = float(models['heated_space'].predict(X)[0])
            building_row['heated_space'] = hs_pred
            building_row['heated_space_encoded'] = hs_pred
        
        # Niveau 4 : IHD
        if 'initial_heat_demand' in models and 'initial_heat_demand' in FEATURES:
            feats = FEATURES['initial_heat_demand']
            X = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
            ihd_pred = float(models['initial_heat_demand'].predict(X)[0])
            return ihd_pred
        return None
    
    # Filtrer les bâtiments avec IHD valide
    df_valid = df_ml[df_ml['initial_heat_demand'] > 100].copy()
    
    # ── Analyse par building_type ──────────────────────────────
    print("=" * 80)
    print("ANALYSE IHD PAR BUILDING_TYPE (cascade production)")
    print("=" * 80)
    
    bt_results = {}
    for bt in sorted(df_valid['building_type'].unique()):
        subset = df_valid[df_valid['building_type'] == bt]
        sample = subset.sample(n=min(sample_per_category, len(subset)), random_state=42)
        
        errors = []
        for _, row in sample.iterrows():
            ihd_pred = predict_ihd_cascade(row)
            if ihd_pred and row['initial_heat_demand'] > 0:
                error = abs(ihd_pred - row['initial_heat_demand']) / row['initial_heat_demand'] * 100
                errors.append(error)
        
        if errors:
            mape = np.mean(errors)
            median_err = np.median(errors)
            bt_results[bt] = {'mape': mape, 'median': median_err, 'n': len(errors)}
            print(f"  {bt:15s} : MAPE={mape:.1f}%, médiane={median_err:.1f}%, n={len(errors)}")
    
    # ── Analyse par initial_heating_system ──────────────────────
    print(f"\n{'='*80}")
    print("ANALYSE IHD PAR INITIAL_HEATING_SYSTEM (cascade production)")
    print("=" * 80)
    
    hs_results = {}
    for hs in sorted(df_valid['initial_heating_system'].unique()):
        subset = df_valid[df_valid['initial_heating_system'] == hs]
        sample = subset.sample(n=min(sample_per_category, len(subset)), random_state=42)
        
        errors = []
        for _, row in sample.iterrows():
            ihd_pred = predict_ihd_cascade(row)
            if ihd_pred and row['initial_heat_demand'] > 0:
                error = abs(ihd_pred - row['initial_heat_demand']) / row['initial_heat_demand'] * 100
                errors.append(error)
        
        if errors:
            mape = np.mean(errors)
            median_err = np.median(errors)
            hs_results[hs] = {'mape': mape, 'median': median_err, 'n': len(errors)}
            print(f"  {hs:30s} : MAPE={mape:.1f}%, médiane={median_err:.1f}%, n={len(errors)}")
    
    return bt_results, hs_results

bt_results, hs_results = analyze_ihd_by_category(
    df_ml, models_production, label_encoders_production, FEATURES_PRODUCTION, 
    sample_per_category=500
)

ANALYSE IHD PAR BUILDING_TYPE (cascade production)
  EFH             : MAPE=30.9%, médiane=21.6%, n=500
  GHD             : MAPE=301.2%, médiane=53.4%, n=500
  GMH             : MAPE=37.4%, médiane=24.7%, n=500
  HH              : MAPE=55.4%, médiane=48.2%, n=7
  Industrie       : MAPE=485.5%, médiane=98.1%, n=500
  MFH             : MAPE=108.1%, médiane=32.9%, n=500
  RH              : MAPE=28.0%, médiane=20.6%, n=500
  Öffentlich      : MAPE=300.9%, médiane=46.7%, n=500

ANALYSE IHD PAR INITIAL_HEATING_SYSTEM (cascade production)
  Fernwärme (Bestand)            : MAPE=42.5%, médiane=24.7%, n=500
  GEH (Bestand)                  : MAPE=139.9%, médiane=33.8%, n=500
  Gaskessel (Bestand)            : MAPE=61.5%, médiane=25.9%, n=500
  Industrie Gasanlage            : MAPE=778.9%, médiane=92.8%, n=500
  Industrie Stromanlage          : MAPE=132.1%, médiane=98.6%, n=218
  Nachtspeicher (Bestand)        : MAPE=277.9%, médiane=55.1%, n=500
  Nahwärme                       : MAPE=39.8%, méd

## TEST INDUSTRIEL — Prédiction complète sur une adresse

In [108]:
def predict_from_profile_production(profile: dict, df_ml: pd.DataFrame, 
                                     models: dict, label_encoders: dict, 
                                     FEATURES: dict) -> dict:
    footprint = profile.get('footprint', {})
    height = profile.get('height', {})
    spatial = profile.get('spatial_context', {})
    geo = profile.get('geocoding', {})
    
    # Matching dans df_ml pour récupérer les features du bloc
    street_name = geo.get('details', {}).get('road', '')
    house_number = str(geo.get('details', {}).get('house_number', '')).strip()
    search = f"{street_name} {house_number}".strip().lower()
    
    def normalize(s):
        if pd.isna(s): return ''
        s = str(s).strip().lower()
        s = s.replace('ß', 'ss').replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue')
        return s
    
    search_norm = normalize(search)
    df_ml['_street_norm'] = df_ml['street'].apply(normalize) if 'street' in df_ml.columns else ''
    match = df_ml[df_ml['_street_norm'] == search_norm]
    
    if len(match) == 0:
        street_only = normalize(street_name)
        match = df_ml[df_ml['_street_norm'].str.startswith(street_only)]
        if len(match) > 1 and footprint.get('floor_area_m2'):
            match = match.copy()
            match['_fa_diff'] = abs(match['floor_area'] - footprint['floor_area_m2'])
            match = match.nsmallest(1, '_fa_diff')
    
    if len(match) == 0:
        print(f"❌ Bâtiment non trouvé dans df_ml")
        df_ml.drop(columns=['_street_norm'], inplace=True, errors='ignore')
        return None
    
    if len(match) > 1:
        match = match.head(1)
    
    # Copier la ligne pour la modifier avec les prédictions
    building_row = match.iloc[0].copy()
    df_ml.drop(columns=['_street_norm'], inplace=True, errors='ignore')
    
    predictions = {}
    
    # ── NIVEAU 1 : Prédictions de base ──────────────────────────
    for target in ['building_type', 'type_of_use', 'construction_year_class']:
        if target in models and target in FEATURES:
            feats = FEATURES[target]
            X_input = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
            pred_encoded = models[target].predict(X_input)[0]
            pred = label_encoders[target].inverse_transform([pred_encoded])[0]
            predictions[target] = pred
            
            # Injecter la PRÉDICTION encodée dans building_row pour la cascade
            building_row[f'{target}_encoded'] = pred_encoded
    
    # ── NIVEAU 2 : Variables dépendantes ────────────────────────
    for target in ['initial_heating_system', 'renovation_state']:
        if target in models and target in FEATURES:
            feats = FEATURES[target]
            X_input = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
            pred_encoded = models[target].predict(X_input)[0]
            pred = label_encoders[target].inverse_transform([pred_encoded])[0]
            predictions[target] = pred
            building_row[f'{target}_encoded'] = pred_encoded
    
    for target in ['number_of_apartments_min', 'number_of_apartments_max']:
        if target in models and target in FEATURES:
            feats = FEATURES[target]
            X_input = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
            pred = float(models[target].predict(X_input)[0])
            predictions[target] = round(pred, 2)
    
    # ── NIVEAU 3 : heated_space ─────────────────────────────────
    if 'heated_space' in models and 'heated_space' in FEATURES:
        feats = FEATURES['heated_space']
        X_input = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
        pred = float(models['heated_space'].predict(X_input)[0])
        predictions['heated_space'] = round(pred, 2)
        # Injecter pour le Niveau 4
        building_row['heated_space'] = pred
        building_row['heated_space_encoded'] = pred
    
    # ── NIVEAU 4 : IHD et construction_year ─────────────────────
    for target in ['initial_heat_demand', 'construction_year']:
        if target in models and target in FEATURES:
            feats = FEATURES[target]
            X_input = pd.DataFrame([building_row[feats].fillna(0)], columns=feats)
            pred = float(models[target].predict(X_input)[0])
            predictions[target] = round(pred, 2)
    
    # ── Post-processing ─────────────────────────────────────────
    bt = predictions.get('building_type', '')
    if bt in ['Industrie', 'GHD', 'Öffentlich']:
        predictions['number_of_apartments_min'] = 0
        predictions['number_of_apartments_max'] = 0
    elif bt == 'EFH':
        predictions['number_of_apartments_min'] = max(1, min(2, round(predictions.get('number_of_apartments_min', 1))))
        predictions['number_of_apartments_max'] = max(1, min(2, round(predictions.get('number_of_apartments_max', 1))))
    elif bt == 'RH':
        predictions['number_of_apartments_min'] = max(1, min(2, round(predictions.get('number_of_apartments_min', 1))))
        predictions['number_of_apartments_max'] = max(1, min(2, round(predictions.get('number_of_apartments_max', 1))))
    elif bt in ['MFH', 'GMH', 'HH']:
        predictions['number_of_apartments_min'] = max(3, round(predictions.get('number_of_apartments_min', 3)))
        predictions['number_of_apartments_max'] = max(predictions['number_of_apartments_min'],
                                                       round(predictions.get('number_of_apartments_max', 3)))
    
    return predictions

if profile:
    print("=" * 70)
    print(f"TEST INDUSTRIEL PRODUCTION — {profile['address']}")
    print("=" * 70)
    
    predictions_ml = predict_from_profile_production(
        profile, df_ml, models_production, label_encoders_production, FEATURES_PRODUCTION
    )
    
    if predictions_ml:
        print("\nPrédictions ML (mode production) :")
        for k, v in predictions_ml.items():
            print(f"  {k:30s} : {v}")
        
        ground_truth = find_ground_truth(profile, df_individual)
        if ground_truth is not None:
            compare_predictions(predictions_ml, ground_truth)

TEST INDUSTRIEL PRODUCTION — Bestenweg 13, Gladbeck, Germany

Prédictions ML (mode production) :
  building_type                  : EFH
  type_of_use                    : Wohngebäude
  construction_year_class        : 2012 - 2023
  initial_heating_system         : Gaskessel (Bestand)
  renovation_state               : renovated
  number_of_apartments_min       : 1
  number_of_apartments_max       : 1
  heated_space                   : 128.37
  initial_heat_demand            : 10966.18
  construction_year              : 2014.46
  Adresse recherchée : 'Bestenweg 13'
  Normalisée : 'bestenweg 13'

✓ Match exact trouvé !
  Adresse dataset : Bestenweg 13
  floor_area → OSM: 83.8 m² | Dataset: 83.8 m² | Écart: 0.0 m²
  building_type : EFH
  heated_space : 145.2396026789541
  initial_heat_demand : 15661.154146605468
  construction_year : 2012
  initial_heating_system : Gaskessel (Bestand)
  renovation_state : partially_renovated
COMPARAISON PRÉDICTIONS vs GROUND TRUTH
                variable

## BENCHMARK AVEC LIGHTGBM, KNN ET RANDOMFOREST

In [ ]:
# =============================================================================
# BENCHMARK MULTI-MODÈLES — Comparaison sur les variables clés
# =============================================================================

import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

# Installer lightgbm si nécessaire : pip install lightgbm

def benchmark_classifiers(X_train, X_test, y_train, y_test, target_name, class_names):
    """Compare plusieurs classifieurs sur une variable."""
    
    results = {}
    
    # XGBoost (baseline)
    model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                               random_state=42, eval_metric='mlogloss', use_label_encoder=False)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    acc = accuracy_score(y_test, model.predict(X_test))
    results['XGBoost'] = acc
    
    # LightGBM
    model_lgb = lgb.LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                     random_state=42, verbose=-1)
    model_lgb.fit(X_train, y_train)
    acc_lgb = accuracy_score(y_test, model_lgb.predict(X_test))
    results['LightGBM'] = acc_lgb
    
    # Random Forest
    model_rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
    model_rf.fit(X_train, y_train)
    acc_rf = accuracy_score(y_test, model_rf.predict(X_test))
    results['RandomForest'] = acc_rf
    
    # KNN
    model_knn = KNeighborsClassifier(n_neighbors=15)
    model_knn.fit(X_train, y_train)
    acc_knn = accuracy_score(y_test, model_knn.predict(X_test))
    results['KNN'] = acc_knn
    
    print(f"\n  {target_name}:")
    for name, acc in sorted(results.items(), key=lambda x: -x[1]):
        marker = " ◄ BEST" if acc == max(results.values()) else ""
        print(f"    {name:15s} : {acc*100:.1f}%{marker}")
    
    return results


def benchmark_regressors(X_train, X_test, y_train, y_test, target_name):
    """Compare plusieurs régresseurs sur une variable."""
    
    results = {}
    
    mask_nz = y_test > 0
    
    def calc_mape(y_true, y_pred):
        m = y_true > 0
        return np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100
    
    # XGBoost (baseline)
    model_hs_cluster = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
    )
    model_hs_cluster.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model_hs_cluster.predict(X_test)
    results['XGBoost'] = {'mape': calc_mape(y_test, y_pred), 'r2': r2_score(y_test, y_pred)}
    
    # LightGBM
    model_lgb = lgb.LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                    random_state=42, verbose=-1)
    model_lgb.fit(X_train, y_train)
    y_pred_lgb = model_lgb.predict(X_test)
    results['LightGBM'] = {'mape': calc_mape(y_test, y_pred_lgb), 'r2': r2_score(y_test, y_pred_lgb)}
    
    # Random Forest
    model_rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
    model_rf.fit(X_train, y_train)
    y_pred_rf = model_rf.predict(X_test)
    results['RandomForest'] = {'mape': calc_mape(y_test, y_pred_rf), 'r2': r2_score(y_test, y_pred_rf)}
    
    # KNN
    model_knn = KNeighborsRegressor(n_neighbors=15)
    model_knn.fit(X_train, y_train)
    y_pred_knn = model_knn.predict(X_test)
    results['KNN'] = {'mape': calc_mape(y_test, y_pred_knn), 'r2': r2_score(y_test, y_pred_knn)}
    
    # Ridge (baseline linéaire)
    model_ridge = Ridge(alpha=1.0)
    model_ridge.fit(X_train, y_train)
    y_pred_ridge = model_ridge.predict(X_test)
    results['Ridge'] = {'mape': calc_mape(y_test, y_pred_ridge), 'r2': r2_score(y_test, y_pred_ridge)}
    
    print(f"\n  {target_name}:")
    for name, metrics in sorted(results.items(), key=lambda x: x[1]['mape']):
        marker = " ◄ BEST MAPE" if metrics['mape'] == min(r['mape'] for r in results.values()) else ""
        print(f"    {name:15s} : MAPE={metrics['mape']:.1f}%, R²={metrics['r2']:.4f}{marker}")
    
    return results


# =============================================================================
# LANCEMENT DU BENCHMARK
# =============================================================================

print("=" * 70)
print("BENCHMARK MULTI-MODÈLES")
print("=" * 70)

all_benchmark_results = {}

# ── Classification : building_type ──────────────────────────────
feats = FEATURES_PRODUCTION['building_type']
le_var = LabelEncoder()
y_var = le_var.fit_transform(df_ml['building_type'].fillna('Unknown'))
X = df_ml[feats].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
all_benchmark_results['building_type'] = benchmark_classifiers(
    X_train, X_test, y_train, y_test, 'building_type', le_var.classes_)

# ── Classification : initial_heating_system ─────────────────────
feats = FEATURES_PRODUCTION['initial_heating_system']
le_var = LabelEncoder()
y_var = le_var.fit_transform(df_ml['initial_heating_system'].fillna('Unknown'))
X = df_ml[feats].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
all_benchmark_results['initial_heating_system'] = benchmark_classifiers(
    X_train, X_test, y_train, y_test, 'initial_heating_system', le_var.classes_)

# ── Classification : type_of_use ─────────────────────
feats = FEATURES_PRODUCTION['type_of_use']
le_var = LabelEncoder()
y_var = le_var.fit_transform(df_ml['type_of_use'].fillna('Unknown'))
X = df_ml[feats].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
all_benchmark_results['type_of_use'] = benchmark_classifiers(
    X_train, X_test, y_train, y_test, 'type_of_use', le_var.classes_)

# ── Classification : construction_year_class ─────────────────────
feats = FEATURES_PRODUCTION['construction_year_class']
le_var = LabelEncoder()
y_var = le_var.fit_transform(df_ml['construction_year_class'].fillna('Unknown'))
X = df_ml[feats].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
all_benchmark_results['construction_year_class'] = benchmark_classifiers(
    X_train, X_test, y_train, y_test, 'construction_year_class', le_var.classes_)

# ── Classification : renovation_state ─────────────────────
feats = FEATURES_PRODUCTION['renovation_state']
le_var = LabelEncoder()
y_var = le_var.fit_transform(df_ml['renovation_state'].fillna('Unknown'))
X = df_ml[feats].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
all_benchmark_results['renovation_state'] = benchmark_classifiers(
    X_train, X_test, y_train, y_test, 'renovation_state', le_var.classes_)


# ── Régression : number_of_apartments_min ───────────
feats = FEATURES_PRODUCTION['number_of_apartments_min']
df_clean = df_ml.dropna(subset=['number_of_apartments_min'])
df_clean = df_clean[df_clean['number_of_apartments_min'] > 0]
X = df_clean[feats].fillna(0)
y = df_clean['number_of_apartments_min']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
all_benchmark_results['number_of_apartments_min'] = benchmark_regressors(
    X_train, X_test, y_train, y_test, 'number_of_apartments_min')

# ── Régression : number_of_apartments_max ───────────
feats = FEATURES_PRODUCTION['number_of_apartments_max']
df_clean = df_ml.dropna(subset=['number_of_apartments_max'])
df_clean = df_clean[df_clean['number_of_apartments_max'] > 0]
X = df_clean[feats].fillna(0)
y = df_clean['number_of_apartments_max']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
all_benchmark_results['number_of_apartments_max'] = benchmark_regressors(
    X_train, X_test, y_train, y_test, 'number_of_apartments_max')

# ── Régression : heated_space (avec features cluster) ───────────
feats = FEATURES_PRODUCTION_CLUSTER['heated_space']
df_clean = df_ml.dropna(subset=['heated_space'])
df_clean = df_clean[df_clean['heated_space'] > 0]
X = df_clean[feats].fillna(0)
y = df_clean['heated_space']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
all_benchmark_results['heated_space'] = benchmark_regressors(
    X_train, X_test, y_train, y_test, 'heated_space')

# ── Régression : initial_heat_demand (avec features cluster) ────
feats = FEATURES_PRODUCTION['initial_heat_demand']
df_clean = df_ml.dropna(subset=['initial_heat_demand'])
df_clean = df_clean[df_clean['initial_heat_demand'] > 0]
q01 = df_clean['initial_heat_demand'].quantile(0.02)
q99 = df_clean['initial_heat_demand'].quantile(0.98)
df_clean = df_clean[(df_clean['initial_heat_demand'] >= q01) & (df_clean['initial_heat_demand'] <= q99)]
X = df_clean[feats].fillna(0)
y = df_clean['initial_heat_demand']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
all_benchmark_results['initial_heat_demand'] = benchmark_regressors(
    X_train, X_test, y_train, y_test, 'initial_heat_demand')

# # ── Régression : tabula_specific_hd ────
# feats = FEATURES_PRODUCTION['tabula_specific_hd']
# df_clean = df_ml.dropna(subset=['tabula_specific_hd'])
# df_clean = df_clean[df_clean['tabula_specific_hd'] > 0]
# X = df_clean[feats].fillna(0)
# y = df_clean['tabula_specific_hd']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# all_benchmark_results['tabula_specific_hd'] = benchmark_regressors(
#     X_train, X_test, y_train, y_test, 'tabula_specific_hd')



BENCHMARK MULTI-MODÈLES

  building_type:
    XGBoost         : 81.3% ◄ BEST
    RandomForest    : 80.6%
    KNN             : 71.7%
    LightGBM        : 71.7%

  initial_heating_system:
    XGBoost         : 72.7% ◄ BEST
    LightGBM        : 72.7%
    RandomForest    : 71.0%
    KNN             : 51.4%

  type_of_use:
    XGBoost         : 94.1% ◄ BEST
    LightGBM        : 94.0%
    RandomForest    : 93.4%
    KNN             : 89.1%

  construction_year_class:
    XGBoost         : 67.7% ◄ BEST
    LightGBM        : 67.6%
    RandomForest    : 65.0%
    KNN             : 50.4%

  renovation_state:
    XGBoost         : 53.2% ◄ BEST
    RandomForest    : 53.1%
    LightGBM        : 53.0%
    KNN             : 43.7%

  number_of_apartments_min:
    RandomForest    : MAPE=29.8%, R²=0.7799 ◄ BEST MAPE
    LightGBM        : MAPE=30.3%, R²=0.7740
    XGBoost         : MAPE=30.6%, R²=0.7682
    KNN             : MAPE=43.0%, R²=0.6265
    Ridge           : MAPE=59.0%, R²=0.6161

  number_

## RÉSUMÉ COMPARATIF FINAL

In [ ]:
# =============================================================================
# RÉSUMÉ COMPARATIF FINAL
# =============================================================================

print(f"\n{'='*70}")
print("RÉSUMÉ — MEILLEUR MODÈLE PAR VARIABLE")
print(f"{'='*70}")

for var, results in all_benchmark_results.items():
    if isinstance(list(results.values())[0], dict):
        best = min(results.items(), key=lambda x: x[1]['mape'])
        print(f"  {var:30s} : {best[0]} (MAPE={best[1]['mape']:.1f}%, R²={best[1]['r2']:.4f})")
    else:
        best = max(results.items(), key=lambda x: x[1])
        print(f"  {var:30s} : {best[0]} ({best[1]*100:.1f}%)")


RÉSUMÉ — MEILLEUR MODÈLE PAR VARIABLE
  building_type                  : XGBoost (81.3%)
  initial_heating_system         : XGBoost (72.6%)
  type_of_use                    : XGBoost (94.1%)
  construction_year_class        : XGBoost (67.7%)
  renovation_state               : RandomForest (53.1%)
  number_of_apartments_min       : RandomForest (MAPE=29.7%, R²=0.7797)
  number_of_apartments_max       : RandomForest (MAPE=25.3%, R²=0.8384)
  heated_space                   : XGBoost (MAPE=15.9%, R²=0.6325)
  initial_heat_demand            : XGBoost (MAPE=30.0%, R²=0.7866)


## ANALYSE IHD — Types TABULA vs Non-TABULA

In [ ]:
# =============================================================================
# ANALYSE IHD — Types TABULA vs Non-TABULA
# =============================================================================

tabula_types = ['EFH', 'RH', 'MFH', 'GMH', 'HH']
non_tabula_types = ['GHD', 'Industrie', 'Öffentlich']

feats = FEATURES_PRODUCTION['initial_heat_demand']

# Préparer le dataset filtré outliers
df_ihd = df_ml.dropna(subset=['initial_heat_demand'])
df_ihd = df_ihd[df_ihd['initial_heat_demand'] > 0]
q01 = df_ihd['initial_heat_demand'].quantile(0.02)
q99 = df_ihd['initial_heat_demand'].quantile(0.98)
df_ihd = df_ihd[(df_ihd['initial_heat_demand'] >= q01) & (df_ihd['initial_heat_demand'] <= q99)]

# Test sur types TABULA uniquement
df_tabula = df_ihd[df_ihd['building_type'].isin(tabula_types)]
X_tab = df_tabula[feats].fillna(0)
y_tab = df_tabula['initial_heat_demand']
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_tab, y_tab, test_size=0.2, random_state=42)

model_tab = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
model_tab.fit(X_train_t, y_train_t, eval_set=[(X_test_t, y_test_t)], verbose=False)
y_pred_t = model_tab.predict(X_test_t)
mask = y_test_t > 0
mape_tab = np.mean(np.abs((y_test_t[mask] - y_pred_t[mask]) / y_test_t[mask])) * 100
r2_tab = r2_score(y_test_t, y_pred_t)

# Test sur types NON-TABULA uniquement
df_non_tab = df_ihd[df_ihd['building_type'].isin(non_tabula_types)]
X_nt = df_non_tab[feats].fillna(0)
y_nt = df_non_tab['initial_heat_demand']
X_train_nt, X_test_nt, y_train_nt, y_test_nt = train_test_split(X_nt, y_nt, test_size=0.2, random_state=42)

model_nt = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
model_nt.fit(X_train_nt, y_train_nt, eval_set=[(X_test_nt, y_test_nt)], verbose=False)
y_pred_nt = model_nt.predict(X_test_nt)
mask_nt = y_test_nt > 0
mape_nt = np.mean(np.abs((y_test_nt[mask_nt] - y_pred_nt[mask_nt]) / y_test_nt[mask_nt])) * 100
r2_nt = r2_score(y_test_nt, y_pred_nt)

# Résumé
print("=" * 60)
print("IHD — TABULA vs Non-TABULA")
print("=" * 60)
print(f"  Types TABULA ({', '.join(tabula_types)}):")
print(f"    n={len(df_tabula)}, MAPE={mape_tab:.1f}%, R²={r2_tab:.4f}")
print(f"  Types Non-TABULA ({', '.join(non_tabula_types)}):")
print(f"    n={len(df_non_tab)}, MAPE={mape_nt:.1f}%, R²={r2_nt:.4f}")
print(f"  Global (tous types):")
print(f"    MAPE={results_production.get('initial_heat_demand', '?')}")

IHD — TABULA vs Non-TABULA
  Types TABULA (EFH, RH, MFH, GMH, HH):
    n=14132, MAPE=12.4%, R²=0.8962
  Types Non-TABULA (GHD, Industrie, Öffentlich):
    n=733, MAPE=45.7%, R²=0.6722
  Global (tous types):
    MAPE=MAPE=14.0%, R²=0.8807


## ANALYSE SHAP 

In [ ]:
# =============================================================================
# ANALYSE SHAP — Importance et impact des features
# =============================================================================

# pip install shap

def shap_analysis(model, X_test, feature_names, target_name, top_n=20):
    """
    Analyse SHAP pour un modèle donné.
    Montre les features les plus influentes et leur direction d'impact.
    """
    print(f"\n{'='*60}")
    print(f"SHAP ANALYSIS — {target_name}")
    print(f"{'='*60}")
    
    # Calculer les SHAP values
    explainer = shap.TreeExplainer(model)
    
    # Utiliser un échantillon pour la vitesse
    sample_size = min(2000, len(X_test))
    X_sample = X_test.sample(n=sample_size, random_state=42) if len(X_test) > sample_size else X_test
    
    shap_values = explainer.shap_values(X_sample)
    
    # Pour la classification multi-classe, shap_values est une liste
    if isinstance(shap_values, list):
        # Anciennes versions SHAP multiclasses
        mean_shap = np.mean(
            [np.mean(np.abs(sv), axis=0) for sv in shap_values],
            axis=0
        )

    elif len(shap_values.shape) == 3:
        # SHAP récent : (samples, features, classes)
        mean_shap = np.mean(np.abs(shap_values), axis=(0, 2))

    else:
        # Régression ou binaire
        mean_shap = np.mean(np.abs(shap_values), axis=0)

    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'mean_shap': mean_shap
    }).sort_values('mean_shap', ascending=False)
    
    # Importance moyenne par feature
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'mean_shap': mean_shap
    }).sort_values('mean_shap', ascending=False)
    
    print(f"\n  Top {top_n} features les plus influentes :")
    for i, (_, row) in enumerate(feature_importance.head(top_n).iterrows()):
        bar = "█" * int(row['mean_shap'] / feature_importance['mean_shap'].max() * 30)
        print(f"    {i+1:2d}. {row['feature']:45s} : {row['mean_shap']:.4f} {bar}")
    
    # Features inutiles (SHAP ~0)
    useless = feature_importance[feature_importance['mean_shap'] < 0.001]
    if len(useless) > 0:
        print(f"\n  Features quasi-inutiles ({len(useless)}) :")
        for _, row in useless.iterrows():
            print(f"    - {row['feature']}")
    
    # # Plot SHAP summary
    # plt.figure(figsize=(12, 8))
    # if isinstance(shap_values, list):
    #     shap.summary_plot(shap_values[0], X_sample, feature_names=feature_names, 
    #                      show=False, max_display=top_n)
    # else:
    #     shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
    #                      show=False, max_display=top_n)
    # plt.title(f'SHAP Summary — {target_name}')
    # plt.tight_layout()
    # plt.savefig(f'shap_{target_name}.png', dpi=150, bbox_inches='tight')
    # plt.show()
    
    return feature_importance



# # ── SHAP pour IHD ──────────────────────────────────────────────
# feats_ihd = FEATURES_PRODUCTION['initial_heat_demand']
# df_clean = df_ml.dropna(subset=['initial_heat_demand'])
# df_clean = df_clean[df_clean['initial_heat_demand'] > 0]
# q01 = df_clean['initial_heat_demand'].quantile(0.02)
# q99 = df_clean['initial_heat_demand'].quantile(0.98)
# df_clean = df_clean[(df_clean['initial_heat_demand'] >= q01) & (df_clean['initial_heat_demand'] <= q99)]
# X_ihd = df_clean[feats_ihd].fillna(0)
# y_ihd = df_clean['initial_heat_demand']
# X_train_ihd, X_test_ihd, y_train_ihd, y_test_ihd = train_test_split(X_ihd, y_ihd, test_size=0.2, random_state=42)

# model_ihd_shap = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
# model_ihd_shap.fit(X_train_ihd, y_train_ihd, verbose=False)

# shap_ihd = shap_analysis(model_ihd_shap, X_test_ihd, feats_ihd, 'initial_heat_demand')

# # ── SHAP pour heated_space ─────────────────────────────────────
# feats_hs = FEATURES_PRODUCTION['heated_space']
# df_clean_hs = df_ml.dropna(subset=['heated_space'])
# df_clean_hs = df_clean_hs[df_clean_hs['heated_space'] > 0]
# X_hs = df_clean_hs[feats_hs].fillna(0)
# y_hs = df_clean_hs['heated_space']
# X_train_hs, X_test_hs, y_train_hs, y_test_hs = train_test_split(X_hs, y_hs, test_size=0.2, random_state=42)

# model_hs_shap = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
# model_hs_shap.fit(X_train_hs, y_train_hs, verbose=False)

# shap_hs = shap_analysis(model_hs_shap, X_test_hs, feats_hs, 'heated_space')

# # ── SHAP pour number_of_apartments_min ───────────
# feats = FEATURES_PRODUCTION['number_of_apartments_min']
# df_clean = df_ml.dropna(subset=['number_of_apartments_min'])
# df_clean = df_clean[df_clean['number_of_apartments_min'] > 0]
# X = df_clean[feats].fillna(0)
# y = df_clean['number_of_apartments_min']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# model_ap_min_shap = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
# model_ap_min_shap.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
# shap_ap_min = shap_analysis(model_ap_min_shap, X_test, feats, 'number_of_apartments_min')

# # ── SHAP pour number_of_apartments_max ───────────
# feats = FEATURES_PRODUCTION['number_of_apartments_max']
# df_clean = df_ml.dropna(subset=['number_of_apartments_max'])
# df_clean = df_clean[df_clean['number_of_apartments_max'] > 0]
# X = df_clean[feats].fillna(0)
# y = df_clean['number_of_apartments_max']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# model_ap_max_shap = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
# model_ap_max_shap.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
# shap_ap_max = shap_analysis(model_ap_max_shap, X_test, feats, 'number_of_apartments_max')

# # ── SHAP pour construction_year ────────────────────────────────────
# feats_cy = FEATURES_PRODUCTION['construction_year']
# df_clean_cy = df_ml.dropna(subset=['construction_year'])
# X_cy = df_clean_cy[feats_cy].fillna(0)
# y_cy = df_clean_cy['construction_year']
# X_train_cy, X_test_cy, y_train_cy, y_test_cy = train_test_split(X_cy, y_cy, test_size=0.2, random_state=42)

# model_cy_shap = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
# model_cy_shap.fit(X_train_cy, y_train_cy, verbose=False)

# shap_cy = shap_analysis(model_cy_shap, X_test_cy, feats_cy, 'construction_year')

# # ── SHAP pour building_type ────────────────────────────────────
# feats_bt = FEATURES_PRODUCTION['building_type']
# le_bt_shap = LabelEncoder()
# y_bt = le_bt_shap.fit_transform(df_ml['building_type'].fillna('Unknown'))
# X_bt = df_ml[feats_bt].fillna(0)
# X_train_bt, X_test_bt, y_train_bt, y_test_bt = train_test_split(X_bt, y_bt, test_size=0.2, random_state=42, stratify=y_bt)

# model_bt_shap = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
#                                     random_state=42, eval_metric='mlogloss', use_label_encoder=False)
# model_bt_shap.fit(X_train_bt, y_train_bt, verbose=False)

# shap_bt = shap_analysis(model_bt_shap, X_test_bt, feats_bt, 'building_type')

# ── SHAP pour renovation_state ─────────────────────
feats = FEATURES_PRODUCTION['renovation_state']
le_var = LabelEncoder()
y_var = le_var.fit_transform(df_ml['renovation_state'].fillna('Unknown'))
X = df_ml[feats].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
model_renov_shap = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                     random_state=42, eval_metric='mlogloss', use_label_encoder=False)
model_renov_shap.fit(X_train, y_train, verbose=False)
shap_renov = shap_analysis(model_renov_shap, X_test, feats, 'renovation_state')

# # ── SHAP pour initial_heating_system ─────────────────────
# feats = FEATURES_PRODUCTION['initial_heating_system']
# le_var = LabelEncoder()
# y_var = le_var.fit_transform(df_ml['initial_heating_system'].fillna('Unknown'))
# X = df_ml[feats].fillna(0)
# X_train, X_test, y_train, y_test = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
# model_ihs_shap = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
#                                      random_state=42, eval_metric='mlogloss', use_label_encoder=False)
# model_ihs_shap.fit(X_train, y_train, verbose=False)

# shap_ihs = shap_analysis(model_ihs_shap, X_test, feats, 'initial_heating_system')

# # ── SHAP pour  type_of_use ─────────────────────
# feats = FEATURES_PRODUCTION['type_of_use']
# le_var = LabelEncoder()
# y_var = le_var.fit_transform(df_ml['type_of_use'].fillna('Unknown'))
# X = df_ml[feats].fillna(0)
# X_train, X_test, y_train, y_test = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
# model_tou_shap = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
#                                      random_state=42, eval_metric='mlogloss', use_label_encoder=False)
# model_tou_shap.fit(X_train, y_train, verbose=False)
# shap_tou = shap_analysis(model_tou_shap, X_test, feats, 'type_of_use')

# # ── SHAP pour construction_year_class ─────────────────────
# feats = FEATURES_PRODUCTION['construction_year_class']
# le_var = LabelEncoder()
# y_var = le_var.fit_transform(df_ml['construction_year_class'].fillna('Unknown'))
# X = df_ml[feats].fillna(0)
# X_train, X_test, y_train, y_test = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
# model_cyc_shap = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
#                                      random_state=42, eval_metric='mlogloss', use_label_encoder=False)
# model_cyc_shap.fit(X_train, y_train, verbose=False)
# shap_cyc = shap_analysis(model_cyc_shap, X_test, feats, 'construction_year_class')


SHAP ANALYSIS — renovation_state

  Top 20 features les plus influentes :
     1. initial_heat_demand_encoded                   : 0.3510 ██████████████████████████████
     2. heated_space_encoded                          : 0.2233 ███████████████████
     3. type_of_use_encoded                           : 0.1450 ████████████
     4. renovation_state__renovated                   : 0.1429 ████████████
     5. renovation_state__partially_renovated         : 0.1357 ███████████
     6. renovation_state__not_renovated               : 0.1236 ██████████
     7. initial_heating_system_encoded                : 0.1137 █████████
     8. building_type_encoded                         : 0.0855 ███████
     9. building_volume                               : 0.0804 ██████
    10. floor_area                                    : 0.0515 ████
    11. height_tif_m                                  : 0.0477 ████
    12. building_type__GHD                            : 0.0466 ███
    13. construction_year_clas

## PHASE — Déclaration LLM 

In [42]:
# =============================================================================
# PHASE 2 — PRÉDICTION LLM 
# =============================================================================
load_dotenv()
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# ── CONFIGURATION DU MODÈLE ─────────────────────────────────
  # True = Ollama local, False = API Mistral
# ── CONFIGURATION DU MODÈLE ─────────────────────────────────
LLM_PROVIDER = "claude"  # "mistral", "groq", "gemini", "local"

if LLM_PROVIDER == "mistral":
    from langchain_mistralai import ChatMistralAI
    llm = ChatMistralAI(model="mistral-medium-latest", 
                        temperature=0.0, 
                        api_key=os.getenv("MISTRAL_API_KEY"))

if LLM_PROVIDER == "groq":
    from langchain_groq import ChatGroq
    llm = ChatGroq(
    model="llama-3.1-8b-instant",  # ou "mixtral-8x7b-32768"
    temperature=0.0,
    api_key=os.getenv("GROQ_API_KEY"))

if LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=0.0,
        api_key=os.getenv("GEMINI_API_KEY"))

if LLM_PROVIDER == "local phi":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="phi4-mini", temperature=0.0, num_ctx=4096)

if LLM_PROVIDER == "local mistral" :
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="mistral", temperature=0.0, num_ctx=4096)

if LLM_PROVIDER == "local gemma" :
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="gemma2:9b", temperature=0.0, num_ctx=4096)

if LLM_PROVIDER == "claude" :
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model="claude-2", temperature=0.0, api_key=os.getenv("CLAUDE_API_KEY"))

print(f"✓ Modèle : {LLM_PROVIDER}")


✓ Modèle : claude


## Extraction des infos profile dans le json

In [35]:
def extract_json_from_response(raw: str) -> dict:
    """
    Extrait un objet JSON d'une réponse LLM, même si le modèle
    ajoute du texte explicatif autour.
    """
    raw = raw.strip()
    
    # Cas 1 : réponse propre qui commence par {
    if raw.startswith('{'):
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            pass
    
    # Cas 2 : JSON dans des backticks markdown
    md_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if md_match:
        try:
            return json.loads(md_match.group(1))
        except json.JSONDecodeError:
            pass
    
    # Cas 3 : trouver le premier { ... } dans le texte
    brace_match = re.search(r'\{[^{}]*\}', raw)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except json.JSONDecodeError:
            pass
    
    # Cas 4 : JSON imbriqué (pour les cas avec des nested braces)
    deep_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if deep_match:
        candidate = deep_match.group(0)
        # Nettoyer les retours à la ligne dans les valeurs
        candidate = re.sub(r'\n\s*', ' ', candidate)
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass
    
    # Cas 5 : extraire les valeurs manuellement par regex
    result = {}
    
    # Chercher des patterns comme "building_type": "MFH" ou building_type : MFH
    for key in ['building_type', 'type_of_use', 'construction_year_class',
                'initial_heating_system', 'renovation_state',
                'number_of_apartments_min', 'number_of_apartments_max',
                'heated_space', 'initial_heat_demand']:
        # Pattern avec guillemets
        pattern = rf'["\']?{key}["\']?\s*[:=]\s*["\']?([^"\',\n\}}]+)["\']?'
        match = re.search(pattern, raw, re.IGNORECASE)
        if match:
            val = match.group(1).strip().rstrip(',').strip('"\'')
            # Essayer de convertir en nombre si possible
            try:
                val = int(val)
            except ValueError:
                try:
                    val = float(val)
                except ValueError:
                    pass
            result[key] = val
    
    if result:
        return result
    
    print(f"    ⚠️ Impossible d'extraire du JSON : {raw[:150]}")
    return {}

## Récupération des few-shots du dataset Digital Twin pour donner des exemples de datas complètes au modèle

In [36]:
# Colonnes du dataset utilisées pour le few-shot
FEATURE_COLS_ALL = [
    'building_type', 'type_of_use', 'construction_year_class',
    'floor_area', 'heated_space', 'initial_heat_demand',
    'initial_heating_system', 'renovation_state',
    'number_of_apartments_min', 'number_of_apartments_max',
    'construction_year', 'solar_potential'
]

def select_training_examples(df: pd.DataFrame, n: int = 8, seed: int = 42) -> pd.DataFrame:
    np.random.seed(seed)
    
    examples = []
    
    # D'abord, garantir au moins 2 exemples par building_type
    for bt, group in df.groupby('building_type'):
        n_take = min(1, len(group))
        examples.append(group.sample(n=n_take, random_state=seed))
    
    # Ensuite, garantir au moins 1 exemple par construction_year_class
    base = pd.concat(examples)
    for cyc, group in df.groupby('construction_year_class'):
        if cyc not in base['construction_year_class'].values:
            examples.append(group.sample(n=1, random_state=seed))
    
    base = pd.concat(examples).drop_duplicates()
    
    # Compléter jusqu'à n
    remaining = df.loc[df.index.difference(base.index)]
    n_extra = n - len(base)
    if n_extra > 0 and len(remaining) > 0:
        extra = remaining.sample(n=min(n_extra, len(remaining)), random_state=seed)
        base = pd.concat([base, extra])
    
    result = base.head(n)
    
    print(f"✓ {len(result)} exemples sélectionnés")
    print(f"  Couverture building_type : {sorted(result['building_type'].unique())}")
    print(f"  Couverture construction_year_class : {sorted(result['construction_year_class'].unique())}")
    print(f"  Répartition building_type :")
    for bt, count in result['building_type'].value_counts().items():
        print(f"    {bt}: {count}")
    
    return result

training_examples = select_training_examples(df_individual, n=8)

✓ 8 exemples sélectionnés
  Couverture building_type : ['EFH', 'GHD', 'GMH', 'HH', 'Industrie', 'MFH', 'RH', 'Öffentlich']
  Couverture construction_year_class : ['1919 - 1948', '1949 - 1978', '1987 - 1990', '2012 - 2023']
  Répartition building_type :
    EFH: 1
    GHD: 1
    GMH: 1
    HH: 1
    Industrie: 1
    MFH: 1
    RH: 1
    Öffentlich: 1


## Génération dynamique des prompts

In [37]:
# =============================================================================
# GÉNÉRATION DYNAMIQUE DES PROMPTS
# =============================================================================

# def build_conso_table(config: dict) -> str:
#     lines = ["| Classe | not_renovated | partially_renovated | renovated |"]
#     lines.append("|--------|--------------|-------------------|-----------|")
#     for classe, vals in config['tabula_specific_hd'].items():
#         lines.append(f"| {classe} | {vals.get('not_renovated', '?')} | {vals.get('partially_renovated', '?')} | {vals.get('renovated', '?')} |")
#     return "\n".join(lines)


# def build_heating_rules(config: dict) -> str:
#     lines = []
#     for bt, dist in config.get("heating_distribution", {}).items():
#         if dist:
#             parts = [f"{sys}: {pct}%" for sys, pct in sorted(dist.items(), key=lambda x: -x[1])]
#             lines.append(f"- {bt} → {', '.join(parts)}")
#     return "\n".join(lines)


# def build_hs_fa_rules(config: dict) -> str:
#     lines = []
#     for bt, ratio in config.get("hs_fa_ratio", {}).items():
#         if ratio > 0:
#             lines.append(f"- {bt}: heated_space ≈ floor_area × {ratio}")
#     return "\n".join(lines)


# def build_building_type_rules(stats: dict) -> str:
#     """Génère les règles de classification data-driven à partir de BUILDING_STATS."""
#     lines = []
#     for bt, s in sorted(stats.items()):
#         fa = s.get('floor_area', {})
#         hs = s.get('heated_space', {})
#         count = s.get('count', 0)
#         share = s.get('share_pct', 0)
#         apt = s.get('apartments', {})      
#         ratio = s.get('ratio_hs_fa', {})
        
#         if not fa:
#             continue
        
#         line = f"{bt} ({share}% of the dataset, {count} buildings) :\n"
#         line += f"  - commun floor_area : {fa.get('p25', '?')} — {fa.get('p75', '?')} m² (médian {fa.get('median', '?')}m²)\n"
#         line += f"  - extreme floor_area : {fa.get('min', '?')} — {fa.get('max', '?')} m²"
        
#         if hs:
#             line += f"\n  - commun heated_space : {hs.get('p25', '?')} — {hs.get('p75', '?')} m² (médian {hs.get('median', '?')}m²)"
        
#         if apt:
#             line += f"\n  - apartments : {apt.get('median_min', '?')} — {apt.get('median_max', '?')}"

#         if ratio:
#             line += f"\n  - ratio commun floors (HS/FA) : {ratio.get('p25', '?')} — {ratio.get('p75', '?')} (médian {ratio.get('median', '?')})"

#         lines.append(line)
    
#     return "\n\n".join(lines)


def generate_prompts(city_config: dict) -> tuple[str, str, str]:
    """
    Génère les 3 system prompts dynamiquement à partir des configs.
    Retourne (prompt_level1, prompt_level2, prompt_level3).
    """
    city = city_config.get("city_name", "Unknown")
    # bt_rules = build_building_type_rules(building_stats)
    # heating_rules = build_heating_rules(city_config)
    # conso_table = build_conso_table(TABULA_SPECIFIC_HD)
    # hs_fa_rules = build_hs_fa_rules(city_config)
    
    # Liste des systèmes de chauffage
    # all_systems = city_config.get('all_heating_systems', [])
    # residential_systems = [s for s in all_systems if not s.startswith('Industrie')]
    # industrial_systems = [s for s in all_systems if s.startswith('Industrie')]

    # ── LEVEL 3 ────────────────────────────────────────────────
    prompt_l3 = f"""You are an expert on building energy performance in {city}.

Predict initial_heat_demand in kWh/year (TOTAL annual heat demand).

TABULA/IWU reference specific consumptions vary by building type, construction period, and renovation state.
The TABULA value will be provided for each building. Use it as a baseline but adjust ±30% based on the building context.

Factors that increase consumption: old building, poor insulation, oil/gas heating, large heated space.
Factors that decrease consumption: recent construction, renovated, heat pump, well-insulated.

RESPOND ONLY with this JSON, nothing else:
{{"initial_heat_demand": X.XX}}"""

    return prompt_l3

# Générer les prompts dynamiques
SYSTEM_PROMPT_LEVEL3 = generate_prompts(
    CITY_CONFIG
)
CITY_CENTER = (CITY_CONFIG.get("center_lat", 0), CITY_CONFIG.get("center_lon", 0))

print(f"\n✓ Pipeline prête pour {CITY_CONFIG.get('city_name', 'Unknown')}")

print(f"  Prompt L3 : {(SYSTEM_PROMPT_LEVEL3)}")



✓ Pipeline prête pour Gelsenkirchen
  Prompt L3 : You are an expert on building energy performance in Gelsenkirchen.

Predict initial_heat_demand in kWh/year (TOTAL annual heat demand).

TABULA/IWU reference specific consumptions vary by building type, construction period, and renovation state.
The TABULA value will be provided for each building. Use it as a baseline but adjust ±30% based on the building context.

Factors that increase consumption: old building, poor insulation, oil/gas heating, large heated space.
Factors that decrease consumption: recent construction, renovated, heat pump, well-insulated.

RESPOND ONLY with this JSON, nothing else:
{"initial_heat_demand": X.XX}


## LANCEMENT PREDICTIONS PAR LLM

In [ ]:
def predict_ihd_llm_v2(profile: dict, ml_predictions: dict, 
                        training_examples: pd.DataFrame, city_config: dict) -> float:
    """
    Prédit IHD via LLM avec les prédictions ML comme input.
    Version améliorée — prompt enrichi avec contexte TABULA + exemples ciblés.
    """
    
    bt = ml_predictions.get('building_type', '')
    cyc = ml_predictions.get('construction_year_class', '')
    rs = ml_predictions.get('renovation_state', 'not_renovated')
    hs = ml_predictions.get('heated_space', 0)
    ihs = ml_predictions.get('initial_heating_system', '')
    footprint = profile.get('footprint', {})
    height = profile.get('height', {})
    fa = footprint.get('floor_area_m2', 0)
    
    # Calculer les fourchettes réelles depuis le dataset pour ce building_type
    same_bt = training_examples[training_examples['building_type'] == bt]
    if len(same_bt) > 10:
        specific_conso = same_bt['initial_heat_demand'] / same_bt['heated_space']
        p10 = specific_conso.quantile(0.1)
        p50 = specific_conso.quantile(0.5)
        p90 = specific_conso.quantile(0.9)
    else:
        p10, p50, p90 = 50, 100, 200

    # Consommation TABULA pour les 3 états de rénovation
    tabula_nr = get_tabula_specific_hd(bt, cyc, 'not_renovated', city_config) or 100
    tabula_pr = get_tabula_specific_hd(bt, cyc, 'partially_renovated', city_config) or 80
    tabula_r = get_tabula_specific_hd(bt, cyc, 'renovated', city_config) or 50
    tabula_current = get_tabula_specific_hd(bt, cyc, rs, city_config) or tabula_nr
    
    # Volume du bâtiment
    h = height.get('height_p90_m', height.get('height_m', 3.4)) if height else 3.4
    volume = fa * h if fa and h else 0
    
    system_prompt = f"""You are a German building energy expert. Predict the TOTAL annual heat demand (IHD) in kWh/year.

RULES:
1. IHD = heated_space × specific_consumption (kWh/m²/year)
2. For {bt} buildings in this dataset, the specific consumption ranges from {p10:.0f} to {p90:.0f} kWh/m²/year (median: {p50:.0f})
3. TABULA reference for {rs}: {tabula_current} kWh/m²/year — this is a theoretical value, real values can differ significantly

KEY FACTORS for specific consumption:
- Renovation state is the STRONGEST factor:
  not_renovated → typically {tabula_nr:.0f} kWh/m²/year
  partially_renovated → typically {tabula_pr:.0f} kWh/m²/year
  renovated → typically {tabula_r:.0f} kWh/m²/year
- Heating system matters: Nachtspeicher/GEH = higher, el.WP = lower, Fernwärme = slightly lower
- Old buildings (pre-1978) with Ölkessel tend to have HIGHER consumption than TABULA suggests
- Recent buildings (post-2000) tend to match TABULA closely

IMPORTANT: Do NOT simply multiply heated_space × TABULA value. 
Look at the examples below and learn the REAL relationship between building characteristics and IHD.

RESPOND ONLY: {{"initial_heat_demand": X.XX}}"""

    # Few-shot — sélectionner des exemples du MÊME building_type
    same_type = training_examples[training_examples['building_type'] == bt]
    if len(same_type) < 5:
        same_type = training_examples
    sample = same_type.sample(n=min(8, len(same_type)), random_state=42)
    
    messages = [SystemMessage(content=system_prompt)]
    
    for _, row in sample.iterrows():
        user_msg = (
            f"building_type: {row['building_type']}, "
            f"construction_year_class: {row['construction_year_class']}, "
            f"renovation_state: {row['renovation_state']}, "
            f"heating_system: {row['initial_heating_system']}, "
            f"floor_area: {row['floor_area']:.1f}m², "
            f"heated_space: {row['heated_space']:.1f}m², "
            f"building_volume: {row['floor_area'] * 3.4:.0f}m³"
        )
        messages.append(HumanMessage(content=user_msg))
        messages.append(AIMessage(content=json.dumps({
            "initial_heat_demand": round(float(row['initial_heat_demand']), 1)
        })))
    
    # Message cible
    target_msg = (
        f"Target building:\n"
        f"  building_type: {bt}\n"
        f"  construction_year_class: {cyc}\n"
        f"  renovation_state: {rs}\n"
        f"  heating_system: {ihs}\n"
        f"  heated_space: {hs:.1f}m²\n"
        f"  floor_area: {fa}m²\n"
        f"  building_volume: {volume:.0f}m³\n"
        f"  floors: {height.get('estimated_floors', '?') if height else '?'}\n"
        f"  distance_center: {profile.get('spatial_context', {}).get('distance_center_km', '?')}km\n"
        f"\nBased on the examples and your expertise, what is the total IHD?"
        f'\n\nRESPOND ONLY: {{"initial_heat_demand": X.XX}}'
    )
    
    messages.append(HumanMessage(content=target_msg))
    
    start = time.time()
    response = llm.invoke(messages)
    elapsed = time.time() - start
    
    raw = response.content.strip()
    prediction = extract_json_from_response(raw)
    
    ihd = prediction.get('initial_heat_demand', None) if prediction else None
    
    if ihd is not None:
        try:
            ihd = float(ihd)
        except (ValueError, TypeError):
            print(f"  ⚠️ IHD non numérique ({ihd}) — fallback TABULA")
            ihd = hs * tabula_current
        
        if ihd < 200 or ihd > 2000000:
            print(f"  ⚠️ IHD aberrant ({ihd}) — fallback sur baseline TABULA")
            ihd = hs * tabula_current
    print(f"✓ IHD LLM v2 — {ihd:.0f} kWh ({elapsed:.1f}s)")
    return ihd

## Test Complet ML (XGBoost) + LLM (Local Mistral)

In [39]:
ihd_llm = predict_ihd_llm_v2(profile, predictions_ml, training_examples, CITY_CONFIG)

✓ IHD LLM v2 — 4810 kWh (0.6s)


## TEST POUR HYPERPARAMETRES VIA OPTUNA

In [ ]:
# =============================================================================
# OPTUNA — Hyperparameter tuning pour building_type
# =============================================================================

# pip install optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

feats = FEATURES_PRODUCTION['building_type']
le_bt_opt = LabelEncoder()
y_bt_opt = le_bt_opt.fit_transform(df_ml['building_type'].fillna('Unknown'))
X_bt_opt = df_ml[feats].fillna(0)

X_train_opt, X_test_opt, y_train_opt, y_test_opt = train_test_split(
    X_bt_opt, y_bt_opt, test_size=0.2, random_state=42, stratify=y_bt_opt
)

def objective_bt(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        'gamma': trial.suggest_float('gamma', 0, 1),
    }
    
    model = xgb.XGBClassifier(
        **params, random_state=42, eval_metric='mlogloss', 
        use_label_encoder=False, verbosity=0
    )
    model.fit(X_train_opt, y_train_opt, eval_set=[(X_test_opt, y_test_opt)], verbose=False)
    y_pred = model.predict(X_test_opt)
    return accuracy_score(y_test_opt, y_pred)

study_bt = optuna.create_study(direction='maximize')
study_bt.optimize(objective_bt, n_trials=50, show_progress_bar=True)

print(f"\n{'='*60}")
print(f"BUILDING_TYPE — Optuna Results")
print(f"{'='*60}")
print(f"  Baseline (default params) : 75.5%")
print(f"  Best Optuna              : {study_bt.best_value*100:.1f}%")
print(f"  Best params : {study_bt.best_params}")

Best trial: 16. Best value: 0.764406: 100%|██████████| 50/50 [19:09<00:00, 23.00s/it]


BUILDING_TYPE — Optuna Results
  Baseline (default params) : 75.5%
  Best Optuna              : 76.4%
  Best params : {'n_estimators': 344, 'max_depth': 10, 'learning_rate': 0.05101357041006488, 'subsample': 0.8229432861272477, 'colsample_bytree': 0.943982810627047, 'min_child_weight': 2, 'reg_alpha': 0.7164935260116321, 'reg_lambda': 0.0016243718295103626, 'gamma': 0.2554716204561269}


In [ ]:
# =============================================================================
# OPTUNA — Tuning pour toutes les variables clés
# =============================================================================

def optuna_tune_classifier(X_train, X_test, y_train, y_test, target_name, n_trials=50):
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        }
        model = xgb.XGBClassifier(**params, random_state=42, eval_metric='mlogloss',
                                    use_label_encoder=False, verbosity=0)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        return accuracy_score(y_test, model.predict(X_test))
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f"  {target_name:30s} : best={study.best_value*100:.1f}% (params: {study.best_params})")
    return study

def optuna_tune_regressor(X_train, X_test, y_train, y_test, target_name, n_trials=50):
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        }
        model = xgb.XGBRegressor(**params, random_state=42, verbosity=0)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        y_pred = model.predict(X_test)
        mask = y_test > 0
        mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
        return mape  # On minimise
    
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f"  {target_name:30s} : best MAPE={study.best_value:.1f}%")
    return study

print("=" * 60)
print("OPTUNA — Tuning de toutes les variables")
print("=" * 60)

studies = {}

# Building type
feats = FEATURES_PRODUCTION['building_type']
le_var = LabelEncoder()
y_var = le_var.fit_transform(df_ml['building_type'].fillna('Unknown'))
X = df_ml[feats].fillna(0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
studies['building_type'] = optuna_tune_classifier(X_tr, X_te, y_tr, y_te, 'building_type')

# Initial heating system
feats = FEATURES_PRODUCTION['initial_heating_system']
le_var = LabelEncoder()
y_var = le_var.fit_transform(df_ml['initial_heating_system'].fillna('Unknown'))
X = df_ml[feats].fillna(0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_var, test_size=0.2, random_state=42, stratify=y_var)
studies['initial_heating_system'] = optuna_tune_classifier(X_tr, X_te, y_tr, y_te, 'initial_heating_system')

# IHD
feats = FEATURES_PRODUCTION_CLUSTER['initial_heat_demand']
df_clean = df_ml.dropna(subset=['initial_heat_demand'])
df_clean = df_clean[df_clean['initial_heat_demand'] > 0]
q01 = df_clean['initial_heat_demand'].quantile(0.02)
q99 = df_clean['initial_heat_demand'].quantile(0.98)
df_clean = df_clean[(df_clean['initial_heat_demand'] >= q01) & (df_clean['initial_heat_demand'] <= q99)]
X = df_clean[feats].fillna(0)
y = df_clean['initial_heat_demand']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
studies['initial_heat_demand'] = optuna_tune_regressor(X_tr, X_te, y_tr, y_te, 'initial_heat_demand')

# Heated space
feats = FEATURES_PRODUCTION_CLUSTER['heated_space']
df_clean = df_ml.dropna(subset=['heated_space'])
df_clean = df_clean[df_clean['heated_space'] > 0]
X = df_clean[feats].fillna(0)
y = df_clean['heated_space']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
studies['heated_space'] = optuna_tune_regressor(X_tr, X_te, y_tr, y_te, 'heated_space')

OPTUNA — Tuning de toutes les variables


Best trial: 5. Best value: 0.764299: 100%|██████████| 50/50 [19:44<00:00, 23.70s/it]


  building_type                  : best=76.4% (params: {'n_estimators': 270, 'max_depth': 10, 'learning_rate': 0.038040970393278244, 'subsample': 0.7118155674694696, 'colsample_bytree': 0.7203488927469888, 'min_child_weight': 3, 'reg_alpha': 0.10946246795654258, 'reg_lambda': 0.6339199631208536})


Best trial: 17. Best value: 0.758133: 100%|██████████| 50/50 [25:41<00:00, 30.82s/it]


  initial_heating_system         : best=75.8% (params: {'n_estimators': 272, 'max_depth': 8, 'learning_rate': 0.08369991669279836, 'subsample': 0.82619017942984, 'colsample_bytree': 0.9987538575415014, 'min_child_weight': 2, 'reg_alpha': 0.45586438934293316, 'reg_lambda': 0.5904891255330235})


Best trial: 42. Best value: 51.0442: 100%|██████████| 50/50 [06:07<00:00,  7.34s/it]


  initial_heat_demand            : best MAPE=51.0%


Best trial: 46. Best value: 16.491: 100%|██████████| 50/50 [04:35<00:00,  5.51s/it] 

  heated_space                   : best MAPE=16.5%


## USE CASE 5 PART 2

## RECUPERATION DES DONNÉES MASTR POUR PV PANNEL

In [ ]:
from open_mastr import Mastr
db = Mastr()
db.download(data=["solar"])  # Télécharge toutes les installations PV

KeyboardInterrupt: 

## EXPLORATION DES DONNÉES MaStR SOLAIRES

In [ ]:
# =============================================================================
# FILTRER LES INSTALLATIONS PV POUR NOS VILLES
# =============================================================================

import sqlite3

db_path = os.path.expanduser("~/.open-MaStR/data/sqlite/open-mastr.db")
conn = sqlite3.connect(db_path)

# Lire les installations solaires avec les colonnes utiles
df_solar = pd.read_sql("""
    SELECT 
        EinheitMastrNummer,
        Gemeinde,
        Ort,
        Postleitzahl,
        Strasse,
        Hausnummer,
        Breitengrad,
        Laengengrad,
        Nettonennleistung,
        Bruttoleistung,
        Inbetriebnahmedatum,
        AnzahlModule,
        Lage,
        Hauptausrichtung,
        EinheitBetriebsstatus
    FROM solar_extended
    WHERE Gemeinde IS NOT NULL
""", conn)

print(f"Total installations PV en Allemagne : {len(df_solar)}")

# Filtrer par ville
cities = ['Gelsenkirchen', 'Düsseldorf', 'Bottrop', 'Gladbeck']

for city in cities:
    subset = df_solar[df_solar['Gemeinde'].str.contains(city, na=False, case=False)]
    print(f"  {city:20s} : {len(subset)} installations PV")

# Garder les installations de nos villes
df_pv = df_solar[df_solar['Gemeinde'].str.contains('|'.join(cities), na=False, case=False)].copy()
print(f"\nTotal pour nos villes : {len(df_pv)}")

# Explorer les données
print(f"\nColonnes avec coordonnées :")
print(f"  Breitengrad (lat) : {df_pv['Breitengrad'].notna().sum()}/{len(df_pv)} valeurs")
print(f"  Laengengrad (lon) : {df_pv['Laengengrad'].notna().sum()}/{len(df_pv)} valeurs")

print(f"\nPuissance (Nettonennleistung) :")
print(df_pv['Nettonennleistung'].describe())

print(f"\nExemples :")
print(df_pv[['Gemeinde', 'Strasse', 'Hausnummer', 'Breitengrad', 'Laengengrad', 'Nettonennleistung']].head(10).to_string())

conn.close()

Total installations PV en Allemagne : 6206925
  Gelsenkirchen        : 6044 installations PV
  Düsseldorf           : 11630 installations PV
  Bottrop              : 6023 installations PV
  Gladbeck             : 3183 installations PV

Total pour nos villes : 26880

Colonnes avec coordonnées :
  Breitengrad (lat) : 865/26880 valeurs
  Laengengrad (lon) : 865/26880 valeurs

Puissance (Nettonennleistung) :
count    26880.000000
mean        10.774176
std        108.818062
min          0.001000
25%          0.800000
50%          5.000000
75%          9.000000
max      15040.000000
Name: Nettonennleistung, dtype: float64

Exemples :
           Gemeinde     Strasse Hausnummer  Breitengrad  Laengengrad  Nettonennleistung
202        Gladbeck         NaN        NaN          NaN          NaN               4.20
322      Düsseldorf         NaN        NaN          NaN          NaN               2.94
500   Gelsenkirchen         NaN        NaN          NaN          NaN               3.00
527        G